# Desynchronised Layer-Wise Hybrid Sparse Training

## Breaking the Synchronisation Bottleneck in Alternating Compressed/DeCompressed Training

**Nouran Mohammad (2644540)** — School of Computer Science & Applied Mathematics, University of the Witwatersrand
Supervised by Dr. Pravesh Ranchod & Dr. Branden Ingram

---

### What this notebook is

This notebook is the **reference implementation and empirical evaluation harness** for the research proposal
*"Efficient Sparse Training"*. It implements the proposed method — an **asynchronous, staggered
Alternating Compressed/DeCompressed (Async-AC/DC) protocol** — and benchmarks it against a
representative cross-section of the sparse-training literature reviewed in the proposal.

### The problem, in one paragraph

Hybrid sparse training (AC/DC, Peste et al., 2021) alternates a network between a *compressed* (sparse) state
and a *decompressed* (dense) state. The sparse phases give memory efficiency; the dense phases give
unconstrained gradient exploration, which is what makes AC/DC more accurate than purely-sparse methods such as
RigL or SNIP. The flaw is that **every layer decompresses at the same instant**. During those windows the
optimiser must hold full dense weights, gradients *and* momentum for the entire network, so the
**peak** memory of AC/DC is identical to that of dense training. Since accelerators fail on *peak*, not
*mean*, memory, AC/DC delivers no usable memory saving during training.

### The proposed fix

Stagger the decompression windows across contiguous blocks of layers so that **at most one block is dense at
any moment**. The peak parameter-state footprint then collapses from

$$M_{\text{peak}}^{\text{sync}} \;=\; M_{\text{dense}} \qquad\text{down to}\qquad
M_{\text{peak}}^{\text{async}} \;=\; M_{\text{sparse}} \;+\; \max_{b}\big(M_b^{\text{dense}} - M_b^{\text{sparse}}\big),$$

while every block still receives **exactly the same number of dense exploratory epochs**. The cost is paid in
optimisation stability (gradient staleness across a sparse/dense boundary), which this notebook addresses with
localised momentum resets, block-local learning-rate warm-up and a compression ramp.

### Headline experimental design

The comparison that matters is not "async vs. sync at unlimited memory" — it is **iso-memory**: given a fixed
accelerator budget, which methods are even *runnable*, and which is the most accurate among those? Under a
budget of roughly one-third of the dense parameter state, AC/DC and dense training are simply **infeasible**,
and the proposed method is the only one in the feasible set that retains dense-quality exploration.

### Contents

| § | Contents |
|---|---|
| 1 | Environment, dependencies, determinism |
| 2 | Experiment configuration |
| 3 | Data pipeline (CIFAR-10 / CIFAR-100) |
| 4 | Architectures (CIFAR ResNets, ResNet-50 for projection) |
| 5 | Sparsity core: masked layers, straight-through estimator, saliency criteria |
| 6 | **Parameter-state memory ledger** (byte-exact analytical model + CUDA validation) |
| 7 | Block partitioning: optimal balanced contiguous partition (dynamic programme) |
| 8 | Phase schedulers: dense, static, AC/DC (sync), **Async-AC/DC (proposed)** |
| 9 | Trainers: the 12 benchmarked methods |
| 10 | Experiment runner |
| 11 | Results: accuracy, peak memory, latency, statistical tests |
| 12 | Iso-memory (feasibility-constrained) comparison — the headline result |
| 13 | Transition-shock analysis: why staggering is *gentler* than synchronising |
| 14 | Ablations: block count, dense duration, ordering, stabilisers |
| 15 | Scaling projection: ResNet-50 / ImageNet peak memory (analytical) |
| 16 | Discussion, limitations, and mapping back to the proposal's objectives |


---
# 1. Environment, Dependencies and Determinism

The notebook is written to run unchanged in three environments:

1. **Google Colab / any CUDA machine** — full experiments, real GPU memory profiling. On Colab the first
   code cell mounts Google Drive and works from a folder there, so the results cache survives a disconnect.
2. **A CPU-only laptop** — use the `smoke` preset; all *memory* results (§6, §12, §15) are analytical and
   therefore fully reproducible without a GPU.
3. **A CUDA workstation** — the `full` preset with multiple seeds.

The cell below installs PyTorch only if it is missing, then pins every source of randomness. We use
`torch.use_deterministic_algorithms(warn_only=True)` rather than the strict form, because a handful of
cuDNN convolution backward kernels have no deterministic implementation; forcing them off would distort the
latency measurements in §11.3, which is a worse scientific trade than a small amount of run-to-run jitter that
we already quantify with multiple seeds.


In [ ]:
# --- Persistent storage on Google Colab -------------------------------------
# Colab's /content disk is wiped when the runtime disconnects. The results cache (S10),
# the figures and the CIFAR download all use relative paths, so moving the working
# directory onto Google Drive is enough for all three to survive a disconnect: an
# interrupted sweep then resumes from the last finished (method, seed) run.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/AsyncACDC"
    os.makedirs(WORK_DIR, exist_ok=True)
    os.chdir(WORK_DIR)
print(f"working dir : {os.getcwd()}")

In [ ]:
# --- Dependency bootstrap -------------------------------------------------
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None, extra_args=()):
    """Import `pkg`, installing `pip_name` first if the import fails."""
    try:
        return importlib.import_module(pkg)
    except ImportError:
        cmd = [sys.executable, "-m", "pip", "install", "-q", pip_name or pkg, *extra_args]
        print(f"[setup] installing {pip_name or pkg} ...")
        subprocess.check_call(cmd)
        return importlib.import_module(pkg)

torch       = _ensure("torch")
torchvision = _ensure("torchvision")
np          = _ensure("numpy")
pd          = _ensure("pandas")
scipy       = _ensure("scipy")
mpl         = _ensure("matplotlib")

import matplotlib.pyplot as plt
print(f"python      : {sys.version.split()[0]}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"numpy       : {np.__version__}")

In [ ]:
# --- Core imports ---------------------------------------------------------
import copy, gc, json, math, os, random, time, warnings
from dataclasses import dataclass, field, asdict, replace
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_CUDA = DEVICE.type == "cuda"

print(f"device      : {DEVICE}")
if ON_CUDA:
    props = torch.cuda.get_device_properties(0)
    print(f"gpu         : {props.name}")
    print(f"gpu memory  : {props.total_memory / 2**30:.2f} GiB")
    print(f"capability  : {props.major}.{props.minor}")
else:
    print("gpu         : none detected -- use the 'smoke' preset for training runs.")
    print("              All analytical memory results (S6, S12, S15) remain exact on CPU.")

In [ ]:
# --- Determinism ----------------------------------------------------------
# cuBLAS is only deterministic with this workspace setting. Without it,
# use_deterministic_algorithms(warn_only=True) merely warns -- and UserWarnings are
# filtered above -- so GPU runs would be silently non-reproducible. It must be set
# before the first cuBLAS call, which is why it lives here and not inside set_seed.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def set_seed(seed: int, deterministic: bool = True) -> None:
    """Pin every RNG we depend on. Called once per (method, seed) run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # warn_only: a few conv-backward kernels have no deterministic variant.
        # Disabling them entirely would bias the latency measurements in S11.3.
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True

def seed_worker(worker_id: int) -> None:
    """DataLoader worker seeding, so augmentation order is reproducible."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(0)
print("RNGs pinned.")

# Global switch. With determinism on, cuDNN cannot autotune its convolution algorithms
# and a few fast kernels are off the table, which costs roughly 20-40% on a small ResNet.
# Turn it off for exploratory sweeps where exact run-to-run reproducibility is not needed;
# leave it on for the runs that go into the report.
DETERMINISTIC = True

In [ ]:
# --- Plot style -----------------------------------------------------------
PALETTE = {
    "dense":      "#4A4A4A",
    "gmp":        "#8C8C8C",
    "snip":       "#7B9E87",
    "grasp":      "#5E8B7E",
    "set":        "#C99A6B",
    "rigl":       "#B5651D",
    "topkast":    "#7A6FA8",
    "str":        "#4E79A7",
    "nm":         "#9C7A97",
    "acdc":       "#C0392B",
    "async_acdc": "#1B7F5C",
    "async_pp":   "#0E5F44",
}
MARKERS = {"acdc": "s", "async_acdc": "o", "async_pp": "D"}

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 1.6,
})

def colour_for(key: str) -> str:
    return PALETTE.get(key, "#666666")

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name: str) -> None:
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, bbox_inches="tight")
    print(f"[fig] {path}")

---
# 2. Experiment Configuration

Every knob lives in a single frozen dataclass so that a run is fully described by one object, which is
serialised into the results table. Three presets are provided:

| preset | epochs | seeds | train subset | intended use |
|---|---|---|---|---|
| `smoke` | 6 | 1 | 2 000 imgs | correctness check; runs on CPU in ~15 min, reduced method set |
| `fast`  | 40 | 1 | full | single-GPU pass, ~1.5 h for all 12 methods on a T4 |
| `full`  | 160 | 3 | full | publication-grade; ~18 h on a T4, ~5 h on an A100 |

**Set `PRESET` in the cell below.** The default is `fast`, which is enough to reproduce every qualitative
claim in the proposal. `smoke` will *not* produce meaningful accuracy numbers — it exists to verify that the
whole pipeline executes end to end — but the memory ledger it produces is exact regardless of preset, because
the ledger depends only on masks and schedules, not on how long we train.

### Notes on the fixed hyper-parameters

* **Optimiser.** SGD, momentum 0.9, weight decay 5e-4, label smoothing 0.0. This matches the CIFAR ResNet
  recipe used by Frankle & Carbin and by Evci et al., so the baselines land where the literature says they
  should.
* **Learning rate.** 0.1 with a cosine schedule and a 2-epoch linear warm-up. Cosine (rather than step) is
  used because AC/DC-style methods are sensitive to where the LR drop lands relative to a phase boundary;
  cosine removes that confound.
* **Which layers are sparsified.** All convolutions and the classifier, **except the stem convolution**.
  Batch-norm affine parameters and biases are never pruned. This is the convention in AC/DC, RigL and
  Top-KAST; the stem is <0.2% of parameters but pruning it is disproportionately damaging, so excluding it is
  standard and applied *identically* to every method here.
* **Sparsity distribution.** Global (a single magnitude threshold across the whole prunable scope) with a
  per-layer density floor of 1%, so no layer can be completely severed.


In [ ]:
@dataclass(frozen=True)
class Config:
    """Complete description of one experimental condition."""

    # ---- data / model
    dataset: str = "cifar10"          # cifar10 | cifar100
    arch: str = "resnet20"            # resnet20 | resnet32 | resnet56
    data_root: str = "./data"
    batch_size: int = 128
    eval_batch_size: int = 512
    num_workers: int = 2
    train_subset: Optional[int] = None

    # ---- optimisation
    epochs: int = 40
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    nesterov: bool = False
    warmup_epochs: int = 2
    grad_clip: Optional[float] = None

    # ---- sparsity
    target_sparsity: float = 0.90
    sparsity_scope: str = "global"    # global | layerwise
    min_layer_density: float = 0.01
    prune_stem: bool = False
    prune_classifier: bool = True

    # ---- AC/DC family schedule (epoch units)
    acdc_warmup_epochs: int = 5       # initial dense exploration before first compression
    acdc_cycle: int = 6               # length of one compress/decompress cycle
    acdc_dense_len: int = 2           # dense epochs per block per cycle
    acdc_final_sparse: int = 11       # terminal sparse fine-tuning window

    # ---- asynchronous protocol (proposed)
    n_blocks: int = 3
    block_partition: str = "balanced" # balanced | stage | uniform
    async_warmup: str = "staggered"   # staggered | sync | snip
    async_order: str = "sequential"   # sequential | sensitivity | reverse
    momentum_reset: str = "local"     # local | global | none
    block_lr_warmup_steps: int = 0    # Async-AC/DC++ stabiliser
    compression_ramp_steps: int = 0   # Async-AC/DC++ stabiliser

    # ---- dynamic sparse baselines
    dst_update_freq: int = 100        # RigL / SET mask-update interval (steps)
    dst_drop_fraction: float = 0.30
    dst_stop_frac: float = 0.75       # stop topology updates after this fraction of training
    topkast_backward_mult: float = 2.0
    topkast_aux_decay: float = 1e-4
    nm_n: int = 1
    nm_m: int = 10

    # ---- memory model
    dtype_bytes: int = 4
    optimizer_states: int = 1         # SGD-momentum keeps one buffer per parameter
    sparse_format: str = "bitmask"    # bitmask | csr

    # ---- bookkeeping
    seed: int = 0
    method: str = "dense"
    eval_every: int = 1
    log_every: int = 100

    def tag(self) -> str:
        return f"{self.method}|{self.arch}|{self.dataset}|s{self.target_sparsity:g}|seed{self.seed}"


PRESETS = {
    "smoke": dict(epochs=6, train_subset=2000, acdc_warmup_epochs=1, acdc_cycle=3,
                  acdc_dense_len=1, acdc_final_sparse=2, warmup_epochs=0, eval_every=1),
    "fast":  dict(epochs=40, acdc_warmup_epochs=5, acdc_cycle=6, acdc_dense_len=2,
                  acdc_final_sparse=11),
    "full":  dict(epochs=160, acdc_warmup_epochs=15, acdc_cycle=15, acdc_dense_len=5,
                  acdc_final_sparse=40),
}

SEEDS_BY_PRESET = {"smoke": (0,), "fast": (0,), "full": (0, 1, 2)}

# >>>>>>>>>>>>>>>>>>>>>>>>  CHOOSE YOUR PRESET HERE  <<<<<<<<<<<<<<<<<<<<<<<<
PRESET = "fast"
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

BASE = Config(**PRESETS[PRESET])
SEEDS = SEEDS_BY_PRESET[PRESET]

# DataLoader workers: Windows spawns rather than forks, and spawned workers inside a
# notebook kernel deadlock on the re-import of __main__. Force the single-process path.
BASE = replace(BASE, num_workers=0 if os.name == "nt" else 2)

if not ON_CUDA and PRESET != "smoke":
    print("!! No CUDA device found. The chosen preset will be extremely slow on CPU.")
    print("!! Consider PRESET = 'smoke', or run this notebook on a GPU runtime.")

print(f"preset      : {PRESET}")
print(f"seeds       : {SEEDS}")
print(f"epochs      : {BASE.epochs}")
print(f"arch/data   : {BASE.arch} / {BASE.dataset}")
print(f"sparsity    : {BASE.target_sparsity:.0%} ({BASE.sparsity_scope})")

---
# 3. Data Pipeline

Standard CIFAR protocol exactly as specified in the proposal (§4.1): per-channel normalisation with the
dataset statistics, random 32x32 crop with 4-pixel reflection padding, and random horizontal flip. No
augmentation is applied at evaluation time.

Two details worth stating because they affect the fairness of the comparison:

* The **same** `DataLoader` seed and worker seeding function are used for every method, so every method sees
  the same batch composition in the same order for a given seed. Differences between methods are therefore
  attributable to the sparsification protocol, not to data ordering.
* `drop_last=True` on the training loader keeps the number of optimiser steps per epoch identical across
  methods, which matters because the AC/DC family schedules phase transitions on step counts.


In [ ]:
DATASET_STATS = {
    "cifar10":  ((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616), 10),
    "cifar100": ((0.5071, 0.4866, 0.4409), (0.2673, 0.2564, 0.2762), 100),
}

def build_dataloaders(cfg: Config) -> Tuple[DataLoader, DataLoader, int]:
    """Return (train_loader, test_loader, n_classes) for the configured dataset."""
    from torchvision import datasets, transforms

    mean, std, n_classes = DATASET_STATS[cfg.dataset]
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    ds_cls = datasets.CIFAR10 if cfg.dataset == "cifar10" else datasets.CIFAR100
    train_ds = ds_cls(cfg.data_root, train=True,  download=True, transform=train_tf)
    test_ds  = ds_cls(cfg.data_root, train=False, download=True, transform=test_tf)

    if cfg.train_subset is not None:
        g = torch.Generator().manual_seed(1234)          # fixed subset across all methods
        idx = torch.randperm(len(train_ds), generator=g)[: cfg.train_subset]
        train_ds = Subset(train_ds, idx.tolist())

    loader_g = torch.Generator();  loader_g.manual_seed(cfg.seed)
    # persistent_workers is deliberately OFF. The loader is memoised across the whole
    # sweep, so a persistent pool stays alive for hours; if the OS reaps a worker (which
    # happens on hosted runtimes under memory pressure) every subsequent method inherits
    # the dead pool and fails with "DataLoader worker exited unexpectedly". Respawning
    # per epoch costs a fraction of a second and makes that failure self-healing.
    common = dict(num_workers=cfg.num_workers, pin_memory=ON_CUDA,
                  persistent_workers=False)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              drop_last=True, worker_init_fn=seed_worker,
                              generator=loader_g, **common)
    test_loader  = DataLoader(test_ds, batch_size=cfg.eval_batch_size, shuffle=False,
                              **common)
    return train_loader, test_loader, n_classes


_LOADER_CACHE: Dict[tuple, tuple] = {}

def get_dataloaders(cfg: Config):
    """Memoised loader construction -- avoids re-reading CIFAR for every method."""
    key = (cfg.dataset, cfg.batch_size, cfg.eval_batch_size, cfg.train_subset,
           cfg.num_workers, cfg.seed)
    if key not in _LOADER_CACHE:
        _LOADER_CACHE[key] = build_dataloaders(cfg)
    return _LOADER_CACHE[key]


_tl, _vl, _nc = get_dataloaders(BASE)
print(f"train batches/epoch : {len(_tl)}   (batch size {BASE.batch_size})")
print(f"test  batches       : {len(_vl)}")
print(f"classes             : {_nc}")

---
# 4. Architectures

The proposal scopes the empirical work to the ResNet family (§4.4, *Architectural Focus*). We use the
**CIFAR ResNets** of He et al. (`ResNet-20/32/56`): three stages of basic residual blocks with
16/32/64 channels. At the two down-sampling points the shortcut is a 1x1 projection convolution with
batch-norm (option B), not the parameter-free zero-padding shortcut (option A) of He et al.'s CIFAR
experiments. Option B keeps every residual block structurally identical, which simplifies the mask registry
and the block partitioner. It adds ~2.8k parameters (272k vs. 270k for ResNet-20), so published option-A
accuracies are a close but not exact reference point.

Two properties make these the right testbed for a *memory-schedule* study rather than, say, a VGG:

1. **A natural block structure.** The three stages give an unambiguous, architecture-respecting partition for
   the staggered schedule, and §7 shows that a parameter-balanced partition is close to but not identical to
   the stage partition — which is itself an interesting result.
2. **A strongly non-uniform parameter distribution.** Stage 3 holds ~76% of the prunable parameters. Any
   naive "one stage at a time" schedule is therefore dominated by that stage, which is precisely why §7
   solves for the *balanced* partition instead.

`ResNet-50` is also constructed (weights untrained) purely so that §15 can compute the ImageNet-scale memory
projection analytically. It is never trained here.


In [ ]:
class BasicBlock(nn.Module):
    """Pre-activation-free basic residual block (He et al., CIFAR variant)."""
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            # Option-B projection shortcut: keeps the module graph uniform, which
            # simplifies both the mask registry and the block partitioner.
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class CifarResNet(nn.Module):
    def __init__(self, depth: int = 20, num_classes: int = 10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "CIFAR ResNet depth must be 6n+2"
        n = (depth - 2) // 6
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)   # stem -- never pruned
        self.bn1   = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, n, 1)
        self.layer2 = self._make_layer(32, n, 2)
        self.layer3 = self._make_layer(64, n, 2)
        self.fc = nn.Linear(64, num_classes)
        self._init_weights()

    def _make_layer(self, planes, blocks, stride):
        layers, strides = [], [stride] + [1] * (blocks - 1)
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0); nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer3(self.layer2(self.layer1(out)))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def build_model(cfg: Config, num_classes: int) -> nn.Module:
    depth = {"resnet20": 20, "resnet32": 32, "resnet56": 56}[cfg.arch]
    return CifarResNet(depth, num_classes)


_m = build_model(BASE, _nc)
_total = sum(p.numel() for p in _m.parameters())
_conv_fc = sum(p.numel() for n, p in _m.named_parameters()
               if p.dim() > 1 and "bn" not in n)
print(f"{BASE.arch}: {_total:,} parameters total, {_conv_fc:,} in conv/fc weight tensors")
for stage in ["conv1", "layer1", "layer2", "layer3", "fc"]:
    n = sum(p.numel() for name, p in _m.named_parameters()
            if name.startswith(stage) and p.dim() > 1)
    print(f"  {stage:<8} {n:>8,}  ({n / _conv_fc:5.1%} of prunable)")
del _m

---
# 5. Sparsity Core

Every method in this notebook is expressed through **one** mechanism, which keeps the comparison honest: no
method gets an implementation advantage the others do not have.

### 5.1 Two masks, not one

A sparse layer carries two binary masks over the same weight tensor:

* $\mathbf{m}^{\text{fwd}}$ — which weights participate in the forward pass. This defines the model that is
  actually evaluated, and therefore the *reported* sparsity.
* $\mathbf{m}^{\text{bwd}}$ — which weights are permitted to receive an update. Always
  $\mathbf{m}^{\text{fwd}} \subseteq \mathbf{m}^{\text{bwd}}$.

For almost every method the two coincide. They differ for **Top-KAST** (Jayakumar et al., 2020), whose whole
idea is a forward set $A$ of size $K$ and a strictly larger backward set $B$ of size $K+M$, and they differ
transiently for **RigL**, which needs the dense gradient at regrowth steps.

### 5.2 The straight-through estimator

The forward pass computes $\widetilde{\mathbf{w}} = \mathbf{m}^{\text{fwd}} \odot \mathbf{w}$. If we let
autograd differentiate that product directly, we get
$\partial L/\partial \mathbf{w} = \mathbf{m}^{\text{fwd}} \odot \partial L / \partial \widetilde{\mathbf{w}}$,
i.e. the gradient is *already* zero outside the forward mask — and RigL's regrowth criterion, Top-KAST's
backward set and SNIP's connection sensitivity all become impossible to compute.

We therefore route the masking through a straight-through estimator,

$$\text{fwd}: \widetilde{\mathbf{w}} = \mathbf{m}\odot\mathbf{w}, \qquad
\text{bwd}: \frac{\partial L}{\partial \mathbf{w}} \;\leftarrow\; \frac{\partial L}{\partial \widetilde{\mathbf{w}}},$$

so `w.grad` always holds the **dense** gradient $\nabla_{\widetilde{\mathbf{w}}} L$. Update legality is then
enforced explicitly, *after* `backward()` and *before* `step()`, by multiplying the gradient with
$\mathbf{m}^{\text{bwd}}$. This single design decision is what lets one code path serve magnitude pruning,
SNIP, GraSP, SET, RigL, Top-KAST, N:M and the entire AC/DC family.

**Important accounting caveat.** Materialising a dense `w.grad` is a *framework* artefact, not a property of
the algorithms. The proposal anticipates exactly this (§4.6, *Framework Memory Overhead*). §6 therefore
measures the memory an efficient implementation *would* use — the standard convention in this literature
(RigL and Top-KAST both report theoretical rather than PyTorch-observed costs) — and §6.4 validates that
analytical model against a real CUDA allocation harness.

### 5.3 Update order

Each optimiser step executes, in this exact order:

1. `loss.backward()` — dense gradients land in `w.grad`.
2. *(method hook)* read dense gradients for topology decisions (RigL, SNIP, GraSP).
3. `w.grad *= m_bwd` — illegal updates are annihilated.
4. `optimizer.step()`.
5. `w.data *= m_bwd` — projection back onto the legal support, which also cancels the weight-decay drift that
   would otherwise pull pruned entries away from zero.

Note that step 5 projects onto $\mathbf{m}^{\text{bwd}}$, **not** $\mathbf{m}^{\text{fwd}}$: under Top-KAST
the weights in $B \setminus A$ are legitimately non-zero even though they sit out the forward pass.


In [ ]:
class _STEMask(torch.autograd.Function):
    """w * m in the forward pass; identity in the backward pass.

    Keeping the backward as identity means `w.grad` receives the *dense* gradient
    dL/d(w*m), which is what RigL's regrowth criterion, Top-KAST's backward set and
    SNIP's connection sensitivity all require. Legality of the update is enforced
    separately by masking `.grad` before `optimizer.step()`.
    """

    @staticmethod
    def forward(ctx, w, m):
        return w * m

    @staticmethod
    def backward(ctx, g):
        return g, None


def ste_mask(w: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    return _STEMask.apply(w, m)


def _effective_weight(module: nn.Module) -> torch.Tensor:
    """The weight tensor the forward pass actually uses.

    Normally the STE-masked weight. STR (S9.5) is the one method that does not use a
    mask at all: when `str_s` is present the layer switches to the soft-threshold
    reparameterisation W = sign(V) * max(|V| - sigmoid(s), 0), which is differentiable
    in both V and s and therefore needs no straight-through approximation.
    """
    s_raw = getattr(module, "str_s", None)
    if s_raw is not None:
        s = torch.sigmoid(s_raw)
        return torch.sign(module.weight) * torch.clamp(module.weight.abs() - s, min=0.0)
    return ste_mask(module.weight, module.mask_fwd)


class MaskedConv2d(nn.Conv2d):
    def _register(self):
        self.register_buffer("mask_fwd", torch.ones_like(self.weight))
        self.register_buffer("mask_bwd", torch.ones_like(self.weight))
        self.str_s = None

    def forward(self, x):
        return self._conv_forward(x, _effective_weight(self), self.bias)


class MaskedLinear(nn.Linear):
    def _register(self):
        self.register_buffer("mask_fwd", torch.ones_like(self.weight))
        self.register_buffer("mask_bwd", torch.ones_like(self.weight))
        self.str_s = None

    def forward(self, x):
        return F.linear(x, _effective_weight(self), self.bias)


def _swap_class(module: nn.Module) -> nn.Module:
    """Re-class a layer in place, preserving its parameters and buffers."""
    if isinstance(module, nn.Conv2d):
        module.__class__ = MaskedConv2d
    elif isinstance(module, nn.Linear):
        module.__class__ = MaskedLinear
    else:
        raise TypeError(type(module))
    module._register()
    return module

In [ ]:
@dataclass
class LayerHandle:
    """Everything the sparsity machinery needs to know about one prunable tensor."""
    name: str
    module: nn.Module
    block: int = -1          # index of the schedule block this layer belongs to
    dense_flops: float = 0.0 # forward MACs at density 1.0 (filled in by S6.3)
    _nnz_f: int = -1         # cached non-zero counts; -1 means "dirty"
    _nnz_b: int = -1

    @property
    def weight(self) -> torch.Tensor:
        return self.module.weight

    @property
    def numel(self) -> int:
        return self.module.weight.numel()

    @property
    def mask_fwd(self) -> torch.Tensor:
        return self.module.mask_fwd

    @property
    def mask_bwd(self) -> torch.Tensor:
        return self.module.mask_bwd

    # Reading .sum().item() forces a host-device synchronisation, so the counts are
    # cached and only recomputed when a mask is known to have changed. Without this the
    # FLOP accounting alone would add two syncs per layer per step.
    def invalidate(self) -> None:
        self._nnz_f = self._nnz_b = -1

    def nnz_fwd(self) -> int:
        if self._nnz_f < 0:
            self._nnz_f = int(self.mask_fwd.sum().item())
        return self._nnz_f

    def nnz_bwd(self) -> int:
        if self._nnz_b < 0:
            self._nnz_b = int(self.mask_bwd.sum().item())
        return self._nnz_b

    def density(self) -> float:
        return self.nnz_fwd() / self.numel

    def set_masks(self, fwd: torch.Tensor, bwd: Optional[torch.Tensor] = None,
                  nnz_hint: Optional[Tuple[int, int]] = None) -> None:
        self.mask_fwd.copy_(fwd)
        self.mask_bwd.copy_(fwd if bwd is None else bwd)
        if nnz_hint is not None:
            self._nnz_f, self._nnz_b = nnz_hint
        else:
            self.invalidate()


class SparseModel:
    """A model plus its prunable-layer registry and the mask update primitives.

    Deliberately a thin wrapper rather than a subclass of nn.Module: every method in
    this notebook manipulates masks through this one object, so the comparison cannot
    be contaminated by per-method implementation differences.
    """

    def __init__(self, model: nn.Module, cfg: Config):
        self.model = model
        self.cfg = cfg
        self.handles: List[LayerHandle] = []
        self._convert()

    # ------------------------------------------------------------------ setup
    def _is_prunable(self, name: str, module: nn.Module) -> bool:
        if not isinstance(module, (nn.Conv2d, nn.Linear)):
            return False
        if name == "conv1" and not self.cfg.prune_stem:      # stem convolution
            return False
        if isinstance(module, nn.Linear) and not self.cfg.prune_classifier:
            return False
        return True

    def _convert(self) -> None:
        for name, module in self.model.named_modules():
            if self._is_prunable(name, module):
                _swap_class(module)
                self.handles.append(LayerHandle(name=name, module=module))

    # ------------------------------------------------------------- properties
    @property
    def prunable_numel(self) -> int:
        return sum(h.numel for h in self.handles)

    @property
    def total_numel(self) -> int:
        return sum(p.numel() for p in self.model.parameters())

    def dense_numel_non_prunable(self) -> int:
        return self.total_numel - self.prunable_numel

    def global_density(self) -> float:
        return sum(h.nnz_fwd() for h in self.handles) / max(self.prunable_numel, 1)

    def global_sparsity(self) -> float:
        return 1.0 - self.global_density()

    def blocks(self) -> List[List[LayerHandle]]:
        n = max(h.block for h in self.handles) + 1
        out: List[List[LayerHandle]] = [[] for _ in range(n)]
        for h in self.handles:
            out[h.block].append(h)
        return out

    # ------------------------------------------------------------- primitives
    def mask_gradients(self) -> None:
        """Step 3 of the update order: annihilate illegal gradient entries."""
        for h in self.handles:
            if h.weight.grad is not None:
                h.weight.grad.mul_(h.mask_bwd)

    def project_weights(self) -> None:
        """Step 5: re-project weights onto the legal support (kills weight-decay drift)."""
        with torch.no_grad():
            for h in self.handles:
                h.weight.mul_(h.mask_bwd)

    def dense_grads(self) -> Dict[str, torch.Tensor]:
        """Snapshot of the *dense* gradients, available thanks to the STE."""
        return {h.name: h.weight.grad.detach().clone()
                for h in self.handles if h.weight.grad is not None}

    def set_all_dense(self, handles: Optional[Sequence[LayerHandle]] = None) -> None:
        for h in (handles if handles is not None else self.handles):
            h.mask_fwd.fill_(1.0)
            h.mask_bwd.fill_(1.0)
            h._nnz_f = h._nnz_b = h.numel          # exact, no sync needed

    def zero_pruned(self, handles: Optional[Sequence[LayerHandle]] = None) -> None:
        with torch.no_grad():
            for h in (handles if handles is not None else self.handles):
                h.weight.mul_(h.mask_bwd)

    def to(self, device):
        self.model.to(device)
        return self

### 5.4 Saliency criteria

Four scoring functions are needed to cover the benchmark set. All of them return a dictionary
`{layer_name: score_tensor}` with the same shape as the corresponding weight, and all of them are consumed by
the *same* top-$k$ selection routine in §5.5 — so differences between methods are differences in the
**score**, not in how the score is turned into a mask.

| criterion | score $s_j$ | used by |
|---|---|---|
| magnitude | $\lvert w_j\rvert$ | GMP, AC/DC, Async-AC/DC, SET (drop), N:M |
| SNIP | $\lvert g_j \odot w_j \rvert$, normalised | SNIP |
| GraSP | $w_j \cdot (\mathbf{H}\mathbf{g})_j$ (GraSP's score, negated) | GraSP |
| gradient | $\lvert g_j \rvert$ | RigL (regrowth), Top-KAST |

**SNIP.** Lee et al. define connection sensitivity as
$g_j = \partial L(\mathbf{c}\odot\mathbf{w})/\partial c_j |_{\mathbf{c}=\mathbf{1}}$, normalised by
$\sum_k |g_k|$. Because $\widetilde{w}_j = c_j w_j$, the chain rule gives
$\partial L/\partial c_j = (\partial L/\partial \widetilde w_j)\, w_j$, so the score is the elementwise
product of the dense gradient and the weight. The normalisation is a monotone transform and does not change
the top-$k$ selection, but we apply it anyway so the reported scores match the paper.

**GraSP.** Wang et al. score a connection by how much removing it *increases* the gradient norm after
pruning, which requires a Hessian-gradient product. We obtain $\mathbf{Hg}$ without ever forming
$\mathbf{H}$ by differentiating $\mathbf{g}^\top \mathbf{g}_{\text{stop}}$ a second time — one extra
backward pass. GraSP's score is $-w_j (\mathbf{Hg})_j$ and it *removes* the highest-scoring connections, i.e. it
keeps the largest $w_j (\mathbf{Hg})_j$. Because the top-$k$ routine keeps the highest scores, we pass it the
negated score $w_j (\mathbf{Hg})_j$.

Both single-shot criteria are averaged over several mini-batches (default 10) rather than the single batch of
the original papers, because at 90%+ sparsity a one-batch estimate is high-variance enough to cost a
measurable amount of final accuracy — and we want the baselines at their strongest.


In [ ]:
@torch.no_grad()
def score_magnitude(sm: SparseModel, **_) -> Dict[str, torch.Tensor]:
    return {h.name: h.weight.detach().abs() for h in sm.handles}


def _accumulate_grads(sm: SparseModel, loader, n_batches: int, device) -> None:
    """Average the dense gradient of the loss over `n_batches` mini-batches."""
    sm.model.zero_grad(set_to_none=True)
    sm.set_all_dense()                       # criteria are evaluated at c = 1
    it, seen = iter(loader), 0
    sm.model.train()
    while seen < n_batches:
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); continue
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(sm.model(x), y) / n_batches
        loss.backward()
        seen += 1


def score_snip(sm: SparseModel, loader=None, device=DEVICE, n_batches: int = 10,
               **_) -> Dict[str, torch.Tensor]:
    """Connection sensitivity |dL/dc_j| = |g_j * w_j|, normalised (Lee et al., 2019)."""
    _accumulate_grads(sm, loader, n_batches, device)
    raw = {h.name: (h.weight.grad.detach() * h.weight.detach()).abs() for h in sm.handles}
    total = sum(v.sum() for v in raw.values()).clamp_min(1e-12)
    sm.model.zero_grad(set_to_none=True)
    return {k: v / total for k, v in raw.items()}


def score_grasp(sm: SparseModel, loader=None, device=DEVICE, n_batches: int = 10,
                temperature: float = 200.0, **_) -> Dict[str, torch.Tensor]:
    """GraSP (Wang et al., 2020), via a double backward.

    GraSP's importance score is S_j = -w_j * (H g)_j, and it REMOVES the highest-scoring
    weights (reference code: keep_masks = score <= threshold). The shared top-k routine
    keeps the HIGHEST scores, so we return the negation, w_j * (H g)_j, which makes top-k
    keep exactly the weights GraSP keeps.

    The Hessian is never formed. We build g with create_graph=True, then differentiate
    <g, g.detach()> to obtain the Hessian-vector product Hg at the cost of one extra
    backward pass.
    """
    sm.set_all_dense()
    sm.model.zero_grad(set_to_none=True)
    weights = [h.weight for h in sm.handles]

    # Pass 1: g = dL/dw, retained in the graph.
    grads = None
    it = iter(loader)
    for _ in range(n_batches):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(sm.model(x) / temperature, y)
        g = torch.autograd.grad(loss, weights, create_graph=True)
        grads = list(g) if grads is None else [a + b for a, b in zip(grads, g)]
    grads = [g / n_batches for g in grads]

    # Pass 2: differentiate <g, stopgrad(g)> -> Hg.
    inner = sum((g * g.detach()).sum() for g in grads)
    hg = torch.autograd.grad(inner, weights)

    # Negated GraSP score: top-k keeps the largest w*Hg == the smallest -w*Hg, as GraSP does.
    scores = {h.name: h.weight.detach() * hv.detach()
              for h, hv in zip(sm.handles, hg)}
    sm.model.zero_grad(set_to_none=True)
    return scores


@torch.no_grad()
def score_gradient(sm: SparseModel, grads: Dict[str, torch.Tensor] = None,
                   **_) -> Dict[str, torch.Tensor]:
    if grads is None:
        return {h.name: h.weight.grad.detach().abs() if h.weight.grad is not None
                else torch.zeros_like(h.weight) for h in sm.handles}
    return {k: v.abs() for k, v in grads.items()}


SALIENCY = {"magnitude": score_magnitude, "snip": score_snip,
            "grasp": score_grasp, "gradient": score_gradient}

### 5.5 Turning scores into masks

Two selection regimes are supported, and the choice is one of the empirical questions the proposal poses
(§4.2.2, step 3: *"the impact of using global versus per-layer masks for this thresholding will be evaluated
empirically"*).

* **Global.** One threshold across the whole selection scope. Lets the algorithm discover a non-uniform
  layer-wise sparsity distribution, which is where most of the accuracy at extreme sparsity comes from — but
  it can starve a small layer entirely, so a **density floor** of 1% per layer is enforced by forcibly
  reinstating that layer's top entries.
* **Layer-wise.** Every layer gets exactly the target density. Robust, never starves a layer, but leaves
  accuracy on the table at high sparsity.

Crucially, for the asynchronous protocol the "global" scope is *global within the block being compressed*.
This is not a simplification — it is a **requirement**: a truly global threshold would need every layer's
magnitudes simultaneously, which reintroduces a synchronisation point. Confining the threshold to the block
keeps the protocol genuinely local. §14 ablates the cost of that restriction, which turns out to be small.


In [ ]:
@torch.no_grad()
def topk_mask_from_scores(scores: Dict[str, torch.Tensor],
                          handles: Sequence[LayerHandle],
                          density: float,
                          scope: str = "global",
                          min_layer_density: float = 0.01) -> Dict[str, torch.Tensor]:
    """Select the highest-scoring `density` fraction of weights.

    scope='global'   : one threshold over all `handles` (a density floor stops any
                       single layer being severed entirely).
    scope='layerwise': every layer independently keeps exactly `density`.
    """
    masks: Dict[str, torch.Tensor] = {}

    if scope == "layerwise":
        for h in handles:
            k = max(1, int(round(density * h.numel)))
            flat = scores[h.name].flatten()
            idx = torch.topk(flat, k, sorted=False).indices
            m = torch.zeros_like(flat); m[idx] = 1.0
            masks[h.name] = m.view_as(h.weight)
        return masks

    # ---- global -----------------------------------------------------------
    total = sum(h.numel for h in handles)
    k_total = max(1, int(round(density * total)))
    flat_all = torch.cat([scores[h.name].flatten() for h in handles])
    idx = torch.topk(flat_all, k_total, sorted=False).indices
    keep = torch.zeros_like(flat_all); keep[idx] = 1.0

    off = 0
    for h in handles:
        masks[h.name] = keep[off: off + h.numel].view_as(h.weight).clone()
        off += h.numel

    # ---- density floor: no layer may fall below min_layer_density ----------
    for h in handles:
        k_min = max(1, int(math.ceil(min_layer_density * h.numel)))
        if int(masks[h.name].sum().item()) < k_min:
            flat = scores[h.name].flatten()
            idx = torch.topk(flat, k_min, sorted=False).indices
            m = torch.zeros_like(flat); m[idx] = 1.0
            masks[h.name] = m.view_as(h.weight)
    return masks


@torch.no_grad()
def apply_masks(sm: SparseModel, masks: Dict[str, torch.Tensor],
                handles: Optional[Sequence[LayerHandle]] = None,
                backward_masks: Optional[Dict[str, torch.Tensor]] = None) -> None:
    for h in (handles if handles is not None else sm.handles):
        bwd = None if backward_masks is None else backward_masks[h.name]
        h.set_masks(masks[h.name], bwd)
    sm.zero_pruned(handles)


@torch.no_grad()
def erk_densities(handles: Sequence[LayerHandle], density: float,
                  erk_power: float = 1.0) -> Dict[str, float]:
    """Erdos-Renyi-Kernel layer densities (Evci et al., 2020).

    Allocates the parameter budget so that a layer's density scales with
    (fan_in + fan_out + kernel dims) / (product of dims), i.e. small layers stay
    denser. Layers that would exceed density 1.0 are clamped and removed from the
    budget, then the allocation is re-solved -- the standard iterative procedure.
    """
    is_dense: set = set()
    while True:
        divisor, rhs, raw = 0.0, 0.0, {}
        for h in handles:
            n = h.numel
            if h.name in is_dense:
                rhs -= (1.0 - density) * n
                continue
            rhs += density * n
            shape = list(h.weight.shape)
            raw[h.name] = (sum(shape) / float(np.prod(shape))) ** erk_power
            divisor += raw[h.name] * n
        eps = rhs / max(divisor, 1e-12)
        max_prob = max(raw.values()) if raw else 0.0
        if max_prob * eps > 1.0:
            worst = max(raw, key=raw.get)
            is_dense.add(worst)
            continue
        break
    return {h.name: (1.0 if h.name in is_dense else min(1.0, raw[h.name] * eps))
            for h in handles}


@torch.no_grad()
def random_masks(handles: Sequence[LayerHandle], densities: Dict[str, float],
                 generator=None) -> Dict[str, torch.Tensor]:
    """Random masks at the given per-layer densities.

    The draw happens on the host and the result is moved to the parameter's device.
    A CPU torch.Generator cannot seed a CUDA tensor -- passing one to a CUDA-device
    torch.rand raises "Expected a 'cuda' device type for generator". Drawing host-side
    also makes the mask identical on CPU and GPU for a given seed, which keeps runs
    comparable across machines.
    """
    out = {}
    for h in handles:
        k = max(1, int(round(densities[h.name] * h.numel)))
        flat = torch.rand(h.numel, device="cpu", generator=generator)
        idx = torch.topk(flat, k, sorted=False).indices
        m = torch.zeros(h.numel, dtype=h.weight.dtype, device="cpu")
        m[idx] = 1.0
        out[h.name] = m.view_as(h.weight).to(h.weight.device)
    return out


@torch.no_grad()
def nm_structured_mask(weight: torch.Tensor, n: int, m: int) -> torch.Tensor:
    """N:M mask (Zhou et al., 2021): keep the N largest of every M consecutive weights.

    Consecutive is taken along the input-channel-major flattening of the output
    filter, which is the layout NVIDIA sparse tensor cores consume.
    """
    flat = weight.detach().abs().reshape(-1)
    pad = (-flat.numel()) % m
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device)])
    groups = flat.view(-1, m)
    idx = torch.topk(groups, n, dim=1, sorted=False).indices
    mask = torch.zeros_like(groups).scatter_(1, idx, 1.0).view(-1)
    return mask[: weight.numel()].view_as(weight)

---
# 6. The Parameter-State Memory Ledger

This section is the analytical heart of the study, so it is worth being precise about what is measured and
why.

### 6.1 What we are counting, and what we are not

Training memory decomposes into three additive pools:

$$M_{\text{total}} = \underbrace{M_{\text{param}}}_{\text{weights + gradients + optimiser state}}
 \;+\; \underbrace{M_{\text{act}}}_{\text{stored activations}}
 \;+\; \underbrace{M_{\text{misc}}}_{\text{data batch, workspace, allocator slack}}$$

The proposal is explicit (§4.3.1) that the staggered schedule targets $M_{\text{param}}$ and that
$M_{\text{act}}$ is expected to be unchanged. That is exactly right: activation memory is a function of the
architecture and the batch size, both of which are held fixed across every method here. We measure
$M_{\text{act}}$ once (§6.5) and report it as a constant additive term, then **compare methods on
$M_{\text{param}}$**, which is the only pool the schedule can influence.

### 6.2 The cost model

For a prunable layer $l$ with $n_l$ weights and $\text{nnz}_l$ non-zeros, in a state
$\phi_l \in \{\text{sparse},\text{dense}\}$, an efficient implementation stores $\text{live}_l$ values, where
$\text{live}_l = n_l$ if dense and $\text{nnz}_l$ if sparse. With $b$ bytes per scalar and $S$ optimiser state
tensors per parameter (SGD-momentum: $S=1$; Adam: $S=2$):

$$M_l \;=\; b\,\text{live}_l\,(\underbrace{1}_{\text{weights}} + \underbrace{1}_{\text{grads}} + S)
\;+\; \underbrace{I_l(\phi_l)}_{\text{index overhead}}$$

The index structure is stored **once** per layer, not once per tensor, because gradients and momentum share
the layer's sparsity pattern. Two formats are supported:

* `bitmask` — one bit per weight: $I_l = n_l/8$ bytes. Format-independent of sparsity; at 90% sparsity on a
  float32 tensor the total is $0.525\,n_l$ bytes per tensor-triple element versus $4 n_l$ dense.
* `csr` — a 32-bit index per non-zero: $I_l = 4\,\text{nnz}_l$. Cheaper only below ~3% density.

`bitmask` is the default because it is what an AC/DC-style implementation would actually use (the mask
already exists) and because it is the *less* favourable of the two for our method at 90% sparsity — we do not
want the headline result to depend on a flattering choice of storage format.

Non-prunable parameters (batch-norm affines, biases, the stem convolution) are always dense and are added to
every method identically.

### 6.3 Why the peak is what matters

$$M_{\text{peak}} = \max_{t} \sum_l M_l\big(\phi_l(t)\big)$$

An accelerator fails when the *instantaneous* allocation exceeds capacity, so the mean is operationally
irrelevant. Under a synchronised schedule every $\phi_l(t)$ flips to dense simultaneously and the maximum is
the full dense cost. Under the staggered schedule, with blocks $\mathcal{B}_1,\dots,\mathcal{B}_B$ and at
most one dense at a time,

$$M_{\text{peak}}^{\text{async}} = M^{\text{sparse}}_{\text{all}}
 + \max_b \sum_{l \in \mathcal{B}_b}\big(M_l^{\text{dense}} - M_l^{\text{sparse}}\big),$$

which is minimised by making the blocks **equal in parameter count** — the optimisation problem solved in §7.
Note this expression also explains why the *mean* memory of the two schedules is identical when the per-block
dense budgets are matched: staggering redistributes the same total dense-time, it does not reduce it. The
entire benefit is in the peak.


In [ ]:
@dataclass
class MemoryBreakdown:
    """Byte-exact parameter-state accounting for one instant in training."""
    weights: float = 0.0
    grads: float = 0.0
    optimizer: float = 0.0
    index: float = 0.0
    non_prunable: float = 0.0

    @property
    def total(self) -> float:
        return self.weights + self.grads + self.optimizer + self.index + self.non_prunable

    @property
    def total_mib(self) -> float:
        return self.total / 2**20

    def as_dict(self) -> dict:
        d = {k: v / 2**20 for k, v in asdict(self).items()}
        d["total"] = self.total_mib
        return d


def index_overhead_bytes(numel: int, nnz: int, fmt: str, dtype_bytes: int) -> float:
    """Bytes to store the sparsity pattern once per layer."""
    if fmt == "bitmask":
        return numel / 8.0
    if fmt == "csr":
        return 4.0 * nnz               # int32 index per non-zero
    raise ValueError(fmt)


def param_state_memory(sm: SparseModel, cfg: Config,
                       dense_flags: Optional[Dict[str, bool]] = None,
                       nnz_override: Optional[Dict[str, int]] = None
                       ) -> MemoryBreakdown:
    """Parameter-state bytes for the model's *current* mask configuration.

    A layer is treated as dense when its forward mask is all-ones (or when
    `dense_flags` says so), in which case no index structure is stored for it.
    """
    b, S = cfg.dtype_bytes, cfg.optimizer_states
    out = MemoryBreakdown()

    for h in sm.handles:
        n = h.numel
        nnz = (nnz_override or {}).get(h.name, h.nnz_bwd())
        is_dense = (dense_flags or {}).get(h.name, nnz >= n)
        live = n if is_dense else nnz
        out.weights   += b * live
        out.grads     += b * live
        out.optimizer += b * live * S
        if not is_dense:
            out.index += index_overhead_bytes(n, nnz, cfg.sparse_format, b)

    npn = sm.dense_numel_non_prunable()
    out.non_prunable = b * npn * (2 + S)
    return out


def dense_reference_memory(sm: SparseModel, cfg: Config) -> MemoryBreakdown:
    """Cost of training the same architecture with no sparsity at all."""
    b, S = cfg.dtype_bytes, cfg.optimizer_states
    n = sm.prunable_numel
    return MemoryBreakdown(weights=b * n, grads=b * n, optimizer=b * n * S,
                           index=0.0,
                           non_prunable=b * sm.dense_numel_non_prunable() * (2 + S))


def fully_sparse_memory(sm: SparseModel, cfg: Config, density: float) -> MemoryBreakdown:
    """Cost when every prunable layer sits at the target density."""
    b, S = cfg.dtype_bytes, cfg.optimizer_states
    out = MemoryBreakdown()
    for h in sm.handles:
        nnz = int(round(density * h.numel))
        out.weights   += b * nnz
        out.grads     += b * nnz
        out.optimizer += b * nnz * S
        out.index     += index_overhead_bytes(h.numel, nnz, cfg.sparse_format, b)
    out.non_prunable = b * sm.dense_numel_non_prunable() * (2 + S)
    return out


class MemoryTracker:
    """Records the parameter-state trace over a run and reports peak/mean."""

    def __init__(self, sm: SparseModel, cfg: Config):
        self.sm, self.cfg = sm, cfg
        self.steps: List[int] = []
        self.epochs: List[float] = []
        self.trace: List[MemoryBreakdown] = []

    def record(self, step: int, epoch_frac: float) -> MemoryBreakdown:
        mb = param_state_memory(self.sm, self.cfg)
        self.steps.append(step); self.epochs.append(epoch_frac); self.trace.append(mb)
        return mb

    @property
    def peak_mib(self) -> float:
        return max((m.total_mib for m in self.trace), default=0.0)

    @property
    def mean_mib(self) -> float:
        return float(np.mean([m.total_mib for m in self.trace])) if self.trace else 0.0

    @property
    def final_mib(self) -> float:
        return self.trace[-1].total_mib if self.trace else 0.0

    def series(self) -> Tuple[np.ndarray, np.ndarray]:
        return np.asarray(self.epochs), np.asarray([m.total_mib for m in self.trace])

### 6.4 Validating the analytical model against real CUDA allocations

An analytical model is only as good as its agreement with hardware. The harness below **actually allocates**
the tensors a memory-efficient implementation would hold — compressed value buffers plus a bit-packed mask
for sparse layers, full tensors for dense ones — and reads
`torch.cuda.max_memory_allocated()`. Agreement within the allocator's rounding granularity (PyTorch's caching
allocator rounds to 512 B blocks) confirms that §6.2 is not an accounting fiction.

On a CPU-only machine the harness reports the analytical figure only and says so explicitly, rather than
silently substituting a number.


In [ ]:
def cuda_allocation_harness(sm: SparseModel, cfg: Config,
                            dense_blocks: Sequence[int]) -> Dict[str, float]:
    """Allocate the tensors an efficient implementation would hold, and measure.

    For every prunable layer we allocate either (a) full weight/grad/momentum tensors
    if its block is currently dense, or (b) compressed value buffers of length nnz plus
    a bit-packed uint8 mask. Returns analytical vs. measured MiB.
    """
    b, S = cfg.dtype_bytes, cfg.optimizer_states
    dense_set = set(dense_blocks)
    dtype = torch.float32

    analytic = 0.0
    for h in sm.handles:
        n = h.numel
        if h.block in dense_set:
            analytic += b * n * (2 + S)
        else:
            nnz = int(round((1 - cfg.target_sparsity) * n))
            analytic += b * nnz * (2 + S) + index_overhead_bytes(n, nnz, cfg.sparse_format, b)
    analytic += b * sm.dense_numel_non_prunable() * (2 + S)
    analytic_mib = analytic / 2**20

    if not ON_CUDA:
        return {"analytic_mib": analytic_mib, "measured_mib": float("nan"),
                "rel_error_pct": float("nan"), "measured": False}

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    buffers = []
    for h in sm.handles:
        n = h.numel
        if h.block in dense_set:
            live = n; store_mask = False
        else:
            live = int(round((1 - cfg.target_sparsity) * n)); store_mask = True
        for _ in range(2 + S):                       # weights, grads, S optimiser states
            buffers.append(torch.empty(live, dtype=dtype, device="cuda"))
        if store_mask and cfg.sparse_format == "bitmask":
            buffers.append(torch.empty((n + 7) // 8, dtype=torch.uint8, device="cuda"))
        elif store_mask:
            buffers.append(torch.empty(live, dtype=torch.int32, device="cuda"))
    npn = sm.dense_numel_non_prunable()
    for _ in range(2 + S):
        buffers.append(torch.empty(npn, dtype=dtype, device="cuda"))

    measured_mib = (torch.cuda.memory_allocated() - base) / 2**20
    del buffers; gc.collect(); torch.cuda.empty_cache()

    err = 100.0 * (measured_mib - analytic_mib) / max(analytic_mib, 1e-9)
    return {"analytic_mib": analytic_mib, "measured_mib": measured_mib,
            "rel_error_pct": err, "measured": True}

### 6.5 Activation memory and theoretical FLOPs

Two auxiliary measurements complete the picture.

**Activation memory** is obtained by profiling one real forward+backward pass with `torch.cuda` peak
statistics, subtracting the parameter state. It is architecture- and batch-determined, hence identical for
all methods; we report it so that readers can compute total footprints, and so the claim "the schedule cannot
change activation memory" is supported by a measurement rather than an assertion.

**Theoretical training FLOPs** follow the convention of Evci et al.: a sparse layer's forward cost scales
with its forward density and the backward cost is $2\times$ the forward cost, scaled by the *backward*
density (which is why Top-KAST is more expensive per step than RigL at the same forward sparsity, and why
dense phases in the AC/DC family cost full dense compute). Per-layer dense MAC counts are captured once with
forward hooks.


In [ ]:
def measure_dense_flops(sm: SparseModel, input_shape=(1, 3, 32, 32)) -> None:
    """Populate handle.dense_flops with per-layer forward MACs at density 1.0."""
    hooks, counts = [], {}

    def mk(name, module):
        def hook(mod, inp, out):
            if isinstance(mod, nn.Conv2d):
                counts[name] = float(mod.weight.numel() * out.shape[-1] * out.shape[-2])
            else:
                counts[name] = float(mod.weight.numel())
        return hook

    for h in sm.handles:
        hooks.append(h.module.register_forward_hook(mk(h.name, h.module)))
    was_training = sm.model.training
    sm.model.eval()
    with torch.no_grad():
        sm.model(torch.zeros(input_shape, device=next(sm.model.parameters()).device))
    for hk in hooks:
        hk.remove()
    sm.model.train(was_training)
    for h in sm.handles:
        h.dense_flops = counts.get(h.name, 0.0)


def step_flops(sm: SparseModel) -> float:
    """Theoretical FLOPs for one forward+backward at the current mask state.

    Forward scales with the forward density; backward is 2x forward and scales with the
    backward density (Evci et al., 2020, Appendix C convention).
    """
    total = 0.0
    for h in sm.handles:
        d_f = h.nnz_fwd() / h.numel
        d_b = h.nnz_bwd() / h.numel
        total += h.dense_flops * d_f + 2.0 * h.dense_flops * d_b
    return total


def measure_activation_memory(model: nn.Module, batch_size: int,
                              shape=(3, 32, 32)) -> float:
    """Peak MiB attributable to activations for one training step."""
    if not ON_CUDA:
        return float("nan")
    model = model.to(DEVICE); model.train()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    x = torch.randn(batch_size, *shape, device=DEVICE)
    y = torch.randint(0, 10, (batch_size,), device=DEVICE)
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    peak = torch.cuda.max_memory_allocated()
    model.zero_grad(set_to_none=True); del x, y, loss
    gc.collect(); torch.cuda.empty_cache()
    return (peak - before) / 2**20

---
# 7. Block Partitioning — Objective 1 of the Proposal

> *"Formulate a mathematical scheduling algorithm to stagger dense exploratory phases systematically across
> discrete neural network layers."*

The scheduling algorithm has two components: **where to cut** (this section) and **when to fire** (§8).

### 7.1 The partition objective

Recall from §6.3 that the peak is

$$M_{\text{peak}}^{\text{async}} = M^{\text{sparse}}_{\text{all}} + \max_b \Delta_b,
\qquad \Delta_b = \sum_{l\in\mathcal{B}_b}\big(M_l^{\text{dense}} - M_l^{\text{sparse}}\big).$$

Because $M_l^{\text{dense}} - M_l^{\text{sparse}}$ is proportional to $n_l$ for a fixed target density,
minimising the peak is exactly the **min-max contiguous partition problem**:

$$\min_{\text{partition into } B \text{ contiguous blocks}} \;\; \max_b \sum_{l\in\mathcal{B}_b} n_l .$$

Contiguity is not an arbitrary restriction — it is what keeps the schedule implementable as a sliding window
over the network's depth and keeps the gradient path between a dense block and its neighbours short.

### 7.2 Why this matters more than it looks

The obvious partition — "one ResNet stage per block" — is badly imbalanced. In ResNet-20, stage 3 holds
about 76% of the prunable parameters, so a stage partition gives $\max_b n_b \approx 0.76\,n$ and a peak
memory only ~21% better than synchronous. The **optimal balanced partition** approaches
$\max_b n_b \approx n/B$, limited only by layer granularity: a block cannot split a layer, and ResNet-20's
largest single layer holds 13.6% of the prunable weights, so $B=3$ reaches 41% rather than 33% (a peak of
48% of dense). Deeper networks partition more evenly. The
partitioner below solves the problem exactly by dynamic programming:

$$D[i][j] \;=\; \min_{k<i} \; \max\big(D[k][j-1],\; \textstyle\sum_{l=k+1}^{i} n_l\big),$$

in $O(L^2 B)$ time, which is instantaneous for any realistic layer count. Ties are broken toward earlier
cuts, which keeps blocks aligned with residual-stage boundaries where possible.


In [ ]:
def optimal_contiguous_partition(sizes: Sequence[int], n_blocks: int) -> List[int]:
    """Exact min-max contiguous partition by dynamic programming.

    Returns a list of block indices, one per element of `sizes`, minimising the
    largest block total. D[i][j] = min over cut positions k of
    max(D[k][j-1], sum(sizes[k:i])), the classic 'painter's partition' recurrence.
    """
    L = len(sizes)
    n_blocks = max(1, min(n_blocks, L))
    prefix = [0] * (L + 1)
    for i, s in enumerate(sizes):
        prefix[i + 1] = prefix[i] + s

    INF = float("inf")
    D = [[INF] * (n_blocks + 1) for _ in range(L + 1)]
    cut = [[0] * (n_blocks + 1) for _ in range(L + 1)]
    D[0][0] = 0.0
    for i in range(1, L + 1):
        for j in range(1, n_blocks + 1):
            for k in range(j - 1, i):
                cost = max(D[k][j - 1], prefix[i] - prefix[k])
                if cost < D[i][j] - 1e-12:
                    D[i][j] = cost; cut[i][j] = k

    bounds, i, j = [], L, n_blocks
    while j > 0:
        k = cut[i][j]; bounds.append((k, i)); i, j = k, j - 1
    bounds.reverse()

    assign = [0] * L
    for b, (s, e) in enumerate(bounds):
        for idx in range(s, e):
            assign[idx] = b
    return assign


def assign_blocks(sm: SparseModel, cfg: Config) -> None:
    """Write handle.block for every prunable layer according to cfg.block_partition."""
    sizes = [h.numel for h in sm.handles]
    B = cfg.n_blocks

    if cfg.block_partition == "balanced":
        assign = optimal_contiguous_partition(sizes, B)
    elif cfg.block_partition == "uniform":
        per = math.ceil(len(sizes) / B)
        assign = [min(i // per, B - 1) for i in range(len(sizes))]
    elif cfg.block_partition == "stage":
        # Residual stages are the top-level modules named layer1, layer2, ... Any other
        # prunable layer (the classifier, or the stem if prune_stem=True) joins its nearest
        # stage: layers before the first stage go to it, layers after a stage go to that
        # stage. Giving the classifier its own block would spend a whole dense turn on
        # ~0.2% of the weights.
        stages = []
        for h in sm.handles:
            root = h.name.split(".")[0]
            if root.startswith("layer") and root not in stages:
                stages.append(root)
        if not stages:
            raise ValueError("stage partition needs top-level modules named layer1, layer2, ...")
        assign, current = [], 0
        for h in sm.handles:
            root = h.name.split(".")[0]
            if root in stages:
                current = stages.index(root)
            assign.append(min(int(current * B / len(stages)), B - 1))
    else:
        raise ValueError(cfg.block_partition)

    for h, b in zip(sm.handles, assign):
        h.block = b


def block_summary(sm: SparseModel) -> pd.DataFrame:
    rows, total = [], sm.prunable_numel
    for b, layers in enumerate(sm.blocks()):
        n = sum(h.numel for h in layers)
        rows.append({"block": b, "n_layers": len(layers), "params": n,
                     "share_%": 100 * n / total,
                     "layers": ", ".join(h.name.replace("layer", "L") for h in layers[:3])
                               + ("..." if len(layers) > 3 else "")})
    return pd.DataFrame(rows)


# ---- demonstrate the partitioner ----------------------------------------------
_probe_model = build_model(BASE, _nc)
_probe = SparseModel(_probe_model, BASE)
_rows = []
for scheme in ["stage", "uniform", "balanced"]:
    assign_blocks(_probe, replace(BASE, block_partition=scheme))
    shares = [sum(h.numel for h in blk) for blk in _probe.blocks()]
    _rows.append({"scheme": scheme,
                  "block sizes": [f"{s:,}" for s in shares],
                  "max block share": f"{max(shares) / _probe.prunable_numel:.1%}",
                  "imbalance (max/mean)": f"{max(shares) / np.mean(shares):.2f}x"})
print(pd.DataFrame(_rows).to_string(index=False))
print()
assign_blocks(_probe, BASE)
print(block_summary(_probe).to_string(index=False))

---
# 8. Phase Schedulers

A schedule is a boolean matrix $P \in \{0,1\}^{E \times B}$ where $P_{t,b}=1$ means "block $b$ is
decompressed during epoch $t$". Every method in the AC/DC family is fully specified by this matrix, which
makes the sync/async contrast a difference in *one object* rather than a difference in two code paths.

### 8.1 The synchronised baseline (Peste et al., 2021)

$$P^{\text{sync}}_{t,b} = \mathbb{1}\big[t < W\big] \;\lor\; \mathbb{1}\big[W \le t < W+M \;\wedge\;
(t-W)\bmod C < \Delta_d\big]$$

— identical for every $b$, which is precisely the synchronisation bottleneck. A dense warm-up of $W$ epochs
precedes the first compression, and a terminal window of $F$ epochs is fully sparse so that the deployed
model is the compressed one.

### 8.2 The proposed staggered schedule

Give block $b$ a phase offset $\phi_b$ and fire the same window at a different time:

$$P^{\text{async}}_{t,b} = \mathbb{1}\big[W \le t < W+M \;\wedge\;
\big((t-W) - \phi_{\pi(b)}\big)\bmod C \in [0,\Delta_d)\big], \qquad
\phi_i = i\left\lfloor C/B \right\rfloor$$

where $\pi$ is a permutation of the blocks (§8.4). The **non-overlap condition** is

$$B\,\Delta_d \le C,$$

which guarantees $\sum_b P^{\text{async}}_{t,b} \le 1$ for all $t$ — at most one block dense at any instant.

### 8.3 What is matched, and what is not — read this before the results

It would be convenient to claim the two schedules are budget-matched. They are not, and the difference runs
*against* the proposed method, so it needs stating up front.

**What is matched.** Both schedules run for the same number of epochs, take the same number of optimiser
steps, see the same data in the same order, and grant each block exactly $\Delta_d$ dense epochs *per cycle*
of the alternating phase.

**What is not matched — and cannot be.** Count total *dense block-epochs*, $\sum_{t,b} P_{t,b}$, which is the
true measure of how much unconstrained exploration a run receives. The non-overlap constraint
$\sum_b P^{\text{async}}_{t,b} \le 1$ caps the staggered schedule at **one** block-epoch per epoch, so over a
run of $E$ epochs ending in $F$ sparse fine-tuning epochs,

$$\sum_{t,b} P^{\text{async}}_{t,b} \;\le\; E - F,
\qquad\text{while}\qquad
\sum_{t,b} P^{\text{sync}}_{t,b} \;=\; B\Big(W + \big\lfloor \tfrac{M}{C} \big\rfloor \Delta_d\Big),$$

which is larger by up to a factor of $B$. The synchronous schedule buys that extra exploration with peak
memory — that is the whole trade. **The staggered method receives strictly less dense exploration than the
baseline it is compared against**, and the cell below prints the realised budgets for both so the reader can
see the size of the handicap rather than take our word for it.

**But the schedule is optimal given the constraint.** The staggered plan keeps exactly one block dense in
every epoch of both the warm-up and the alternating window, so it *attains* the ceiling $E-F$. No schedule
that holds at most one block dense at a time can extract more dense exploration from the same number of
epochs. The right way to read the comparison is therefore *exploration per unit of peak memory*, which the
summary below reports, and on which the staggered schedule is roughly $B$ times more efficient.

The warm-up is where most of the handicap accumulates: the synchronous protocol warms up with the whole
network dense for $W$ epochs, and doing that asynchronously would reintroduce the exact peak we are removing.
Two alternatives are implemented for the §14 ablations: `async_warmup='snip'` initialises the masks from
block-local connection sensitivity and skips the dense warm-up entirely, and `async_warmup='sync'` restores
the synchronous warm-up, which trades the memory guarantee for exploration parity.

### 8.4 Block ordering

`sequential` traverses blocks shallow-to-deep, `reverse` deep-to-shallow, and `sensitivity` orders blocks by
mean SNIP connection sensitivity so that the blocks that gain most from dense exploration are decompressed
first in each cycle. §14.3 ablates the three.


In [ ]:
@dataclass
class PhasePlan:
    """dense[t, b] == True  <=>  block b is decompressed during epoch t."""
    dense: np.ndarray
    label: str = ""

    @property
    def epochs(self) -> int: return self.dense.shape[0]

    @property
    def n_blocks(self) -> int: return self.dense.shape[1]

    @property
    def max_concurrent(self) -> int: return int(self.dense.sum(axis=1).max())

    def dense_epochs_per_block(self) -> np.ndarray: return self.dense.sum(axis=0)

    def block_dense_epochs(self) -> int:
        """Total block-epochs of dense compute -- the exploration budget."""
        return int(self.dense.sum())

    def transitions(self, t: int) -> List[Tuple[int, str]]:
        """Blocks changing state entering epoch t: (block, 'decompress'|'compress')."""
        prev = self.dense[t - 1] if t > 0 else np.zeros(self.n_blocks, bool)
        out = []
        for b in range(self.n_blocks):
            if self.dense[t, b] and not prev[b]:
                out.append((b, "decompress"))
            elif prev[b] and not self.dense[t, b]:
                out.append((b, "compress"))
        return out

    def final_transitions(self) -> List[Tuple[int, str]]:
        """Blocks still dense at the end must be compressed before deployment."""
        return [(b, "compress") for b in range(self.n_blocks) if self.dense[-1, b]]


def _middle_window(cfg: Config) -> Tuple[int, int]:
    W = min(cfg.acdc_warmup_epochs, cfg.epochs)
    Fz = min(cfg.acdc_final_sparse, max(cfg.epochs - W, 0))
    M = max(cfg.epochs - W - Fz, 0)
    return W, M


def plan_always_dense(cfg: Config) -> PhasePlan:
    return PhasePlan(np.ones((cfg.epochs, cfg.n_blocks), bool), "dense")


def plan_always_sparse(cfg: Config) -> PhasePlan:
    return PhasePlan(np.zeros((cfg.epochs, cfg.n_blocks), bool), "sparse")


def plan_sync_acdc(cfg: Config) -> PhasePlan:
    W, M = _middle_window(cfg)
    P = np.zeros((cfg.epochs, cfg.n_blocks), bool)
    P[:W, :] = True                                        # synchronous dense warm-up
    for t in range(W, W + M):
        if (t - W) % cfg.acdc_cycle < cfg.acdc_dense_len:
            P[t, :] = True                                 # every block, same instant
    return PhasePlan(P, "acdc-sync")


def plan_async_acdc(cfg: Config, order: Optional[Sequence[int]] = None) -> PhasePlan:
    W, M = _middle_window(cfg)
    B = cfg.n_blocks
    # Non-overlap condition (S8.2). Without it two blocks are dense at once and the
    # peak-memory guarantee silently breaks -- e.g. n_blocks=5 with the 'fast' cycle of 6.
    if B * cfg.acdc_dense_len > cfg.acdc_cycle:
        raise ValueError(
            f"n_blocks * acdc_dense_len = {B} * {cfg.acdc_dense_len} = {B * cfg.acdc_dense_len} "
            f"exceeds acdc_cycle = {cfg.acdc_cycle}, so blocks would be dense at the same time. "
            f"Set acdc_cycle >= {B * cfg.acdc_dense_len} or shorten acdc_dense_len.")
    order = list(order) if order is not None else list(range(B))
    P = np.zeros((cfg.epochs, B), bool)

    # ---- warm-up ----------------------------------------------------------
    if cfg.async_warmup == "sync":
        P[:W, :] = True
    elif cfg.async_warmup == "staggered" and W > 0:
        edges = np.linspace(0, W, B + 1).round().astype(int)
        for i, b in enumerate(order):
            P[edges[i]: edges[i + 1], b] = True
    # 'snip': no dense warm-up at all; masks come from block-local sensitivity.

    # ---- staggered alternating phase --------------------------------------
    offset = cfg.acdc_cycle // B
    for t in range(W, W + M):
        u = (t - W) % cfg.acdc_cycle
        for i, b in enumerate(order):
            start = i * offset
            if start <= u < start + cfg.acdc_dense_len:
                P[t, b] = True
    return PhasePlan(P, "acdc-async")


def describe_plan(plan: PhasePlan, cfg: Config) -> dict:
    W, M = _middle_window(cfg)
    ceiling = W + M                      # one block-epoch per epoch is the async maximum
    constrained = plan.max_concurrent <= 1
    return {"schedule": plan.label,
            "max concurrent dense blocks": plan.max_concurrent,
            "dense block-epochs": plan.block_dense_epochs(),
            "per-block dense epochs": plan.dense_epochs_per_block().tolist(),
            "one-block ceiling (E-F)": ceiling,
            "attains ceiling?": ("yes" if plan.block_dense_epochs() >= ceiling else "no")
                                if constrained else "n/a -- unconstrained",
            "non-overlap": bool(cfg.n_blocks * cfg.acdc_dense_len <= cfg.acdc_cycle)}


# ---- visualise the two schedules side by side --------------------------------
_sync = plan_sync_acdc(BASE)
_async = plan_async_acdc(BASE)
print(pd.DataFrame([describe_plan(_sync, BASE), describe_plan(_async, BASE)]).to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(9, 3.4), sharex=True)
for ax, plan, title in zip(axes, [_sync, _async],
                           ["Synchronised AC/DC (Peste et al., 2021)",
                            "Asynchronous staggered AC/DC (proposed)"]):
    ax.imshow(plan.dense.T, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=1,
              interpolation="nearest", origin="lower",
              extent=[0, plan.epochs, -0.5, plan.n_blocks - 0.5])
    ax.set_yticks(range(plan.n_blocks))
    ax.set_yticklabels([f"block {b}" for b in range(plan.n_blocks)])
    ax.set_title(title, loc="left")
    ax.grid(False)
axes[1].set_xlabel("epoch")
fig.suptitle("Decompression schedule  (red = dense, green = sparse)", y=1.02, fontsize=10)
fig.tight_layout(); save_fig(fig, "01_schedules"); plt.show()

### 8.5 The memory consequence, before any training

The schedule matrix alone determines the parameter-state trace, so the central memory claim can be verified
*analytically* — no GPU, no training, no stochasticity. The figure below overlays the peak-memory trajectory
implied by each schedule.


In [ ]:
def simulate_memory_trace(sm: SparseModel, cfg: Config, plan: PhasePlan) -> np.ndarray:
    """Parameter-state MiB per epoch implied by a phase plan (no training required)."""
    b, S = cfg.dtype_bytes, cfg.optimizer_states
    d = 1.0 - cfg.target_sparsity
    blocks = sm.blocks()
    npn_bytes = b * sm.dense_numel_non_prunable() * (2 + S)

    trace = np.zeros(plan.epochs)
    for t in range(plan.epochs):
        total = npn_bytes
        for bi, layers in enumerate(blocks):
            for h in layers:
                n = h.numel
                if plan.dense[t, bi]:
                    total += b * n * (2 + S)
                else:
                    nnz = int(round(d * n))
                    total += b * nnz * (2 + S) + index_overhead_bytes(n, nnz, cfg.sparse_format, b)
        trace[t] = total / 2**20
    return trace


_sm_probe = SparseModel(build_model(BASE, _nc), BASE)
assign_blocks(_sm_probe, BASE)

_tr_sync  = simulate_memory_trace(_sm_probe, BASE, _sync)
_tr_async = simulate_memory_trace(_sm_probe, BASE, _async)
_dense_mib  = dense_reference_memory(_sm_probe, BASE).total_mib
_sparse_mib = fully_sparse_memory(_sm_probe, BASE, 1 - BASE.target_sparsity).total_mib

fig, ax = plt.subplots(figsize=(8.4, 3.2))
ax.step(range(BASE.epochs), _tr_sync,  where="post", color=PALETTE["acdc"],
        label=f"AC/DC (sync)   peak {_tr_sync.max():.2f} MiB")
ax.step(range(BASE.epochs), _tr_async, where="post", color=PALETTE["async_acdc"],
        label=f"Async-AC/DC    peak {_tr_async.max():.2f} MiB")
ax.axhline(_dense_mib,  ls="--", lw=1, color=PALETTE["dense"], label=f"dense training ({_dense_mib:.2f} MiB)")
ax.axhline(_sparse_mib, ls=":",  lw=1, color="#777", label=f"fully sparse floor ({_sparse_mib:.2f} MiB)")
ax.set_xlabel("epoch"); ax.set_ylabel("parameter-state memory (MiB)")
ax.set_title("Analytical parameter-state footprint over training", loc="left")
ax.legend(loc="center right")
fig.tight_layout(); save_fig(fig, "02_memory_trace_analytical"); plt.show()

_be_sync, _be_async = _sync.block_dense_epochs(), _async.block_dense_epochs()

print(f"peak reduction vs sync AC/DC : {100 * (1 - _tr_async.max() / _tr_sync.max()):.1f}%")
print(f"mean memory  sync / async    : {_tr_sync.mean():.2f} / {_tr_async.mean():.2f} MiB")
print(f"dense-delta reduction factor : "
      f"{(_tr_sync.max() - _sparse_mib) / max(_tr_async.max() - _sparse_mib, 1e-9):.2f}x"
      f"   (theoretical ceiling = n_blocks = {BASE.n_blocks})")
print()
print("Exploration budget -- the cost side of the trade (see S8.3):")
print(f"  dense block-epochs       sync {_be_sync:4d}   async {_be_async:4d}"
      f"   ({100 * _be_async / max(_be_sync, 1):.0f}% of the baseline)")
print(f"  per peak MiB             sync {_be_sync / _tr_sync.max():5.1f}   "
      f"async {_be_async / _tr_async.max():5.1f}"
      f"   ({(_be_async / _tr_async.max()) / (_be_sync / _tr_sync.max()):.2f}x more"
      f" exploration per unit of peak memory)")

---
# 9. Methods

Each method is a small object implementing six optional hooks. The training loop (§9.9) is shared, so no
method can gain an advantage from a different loop, a different optimiser, a different LR schedule or a
different data order.

```
setup(sm, cfg, loader)          -> initial masks
on_epoch_start(sm, cfg, epoch)  -> phase transitions (AC/DC family)
on_after_backward(sm, cfg, step) -> read dense gradients before they are masked (RigL)
on_step_end(sm, cfg, step)      -> topology updates (SET, RigL, N:M)
extra_loss(sm, cfg)             -> auxiliary regularisers (Top-KAST, STR)
finalize(sm, cfg)               -> guarantee the deployed model meets the target sparsity
```

### 9.1 Optimiser plumbing: per-block parameter groups

The optimiser is constructed with **one parameter group per block**, plus a group for everything
non-prunable. This costs nothing for the methods that do not use it and is what makes two of the proposed
stabilisers possible:

* **Localised momentum reset.** When block $b$ decompresses, the momentum buffers of *only that block* are
  zeroed. Peste et al. reset globally because their transitions are global; the asynchronous protocol must
  reset locally or it would destroy the accumulated velocity of blocks that are mid-sparse-phase.
* **Block-local LR warm-up.** A per-group learning-rate multiplier ramps from a small value back to 1 over
  the first steps after decompression, so the newly revived weights — which have no meaningful gradient
  history — do not receive a full-size update on their first step.


In [ ]:
def build_optimizer(sm: SparseModel, cfg: Config) -> torch.optim.Optimizer:
    """SGD with one parameter group per block, plus a group for everything else.

    The per-block grouping is what enables localised momentum resets and block-local
    learning-rate warm-up without touching the other blocks' optimiser state.
    """
    prunable_ids = {id(h.weight) for h in sm.handles}
    groups = []
    for b, layers in enumerate(sm.blocks()):
        params = [h.weight for h in layers]
        if params:
            groups.append({"params": params, "block": b, "lr_mult": 1.0,
                           "weight_decay": cfg.weight_decay, "name": f"block{b}"})
    rest = [p for p in sm.model.parameters() if id(p) not in prunable_ids]
    groups.append({"params": rest, "block": -1, "lr_mult": 1.0,
                   "weight_decay": cfg.weight_decay, "name": "other"})

    return torch.optim.SGD(groups, lr=cfg.lr, momentum=cfg.momentum,
                           weight_decay=cfg.weight_decay, nesterov=cfg.nesterov)


def lr_at(cfg: Config, step: int, total_steps: int, steps_per_epoch: int) -> float:
    """Linear warm-up then cosine decay, evaluated per step."""
    warm = cfg.warmup_epochs * steps_per_epoch
    if warm > 0 and step < warm:
        return cfg.lr * (step + 1) / warm
    prog = (step - warm) / max(total_steps - warm, 1)
    return 0.5 * cfg.lr * (1.0 + math.cos(math.pi * min(prog, 1.0)))


def set_lr(opt: torch.optim.Optimizer, base_lr: float) -> None:
    for g in opt.param_groups:
        g["lr"] = base_lr * g.get("lr_mult", 1.0)


@torch.no_grad()
def reset_momentum(opt: torch.optim.Optimizer, params: Iterable[torch.Tensor]) -> None:
    """Zero the momentum buffers of specific parameters (localised reset)."""
    targets = {id(p) for p in params}
    for g in opt.param_groups:
        for p in g["params"]:
            if id(p) in targets and p in opt.state and "momentum_buffer" in opt.state[p]:
                buf = opt.state[p]["momentum_buffer"]
                if buf is not None:
                    buf.zero_()


@torch.no_grad()
def mask_momentum(opt: torch.optim.Optimizer, sm: SparseModel,
                  handles: Sequence[LayerHandle]) -> None:
    """Zero momentum entries that correspond to newly-pruned weights.

    Without this, a pruned weight keeps a stale velocity that would immediately drag it
    off zero on the next step, silently violating the sparsity constraint.
    """
    by_id = {id(h.weight): h for h in handles}
    for g in opt.param_groups:
        for p in g["params"]:
            h = by_id.get(id(p))
            if h is not None and p in opt.state and "momentum_buffer" in opt.state[p]:
                buf = opt.state[p]["momentum_buffer"]
                if buf is not None:
                    buf.mul_(h.mask_bwd)

### 9.2 Method interface and the static baselines

**Dense** is the unpruned reference: the accuracy ceiling and the memory ceiling.

**GMP** (gradual magnitude pruning, Zhu & Gupta / Han et al. lineage) ramps sparsity from 0 to the target
with the standard cubic schedule
$s_t = s_f\big(1-(1-\tfrac{t-t_0}{n})^3\big)$, then fine-tunes. It represents the "prune during training,
never explore sparsely" family and is a strong accuracy baseline — but it is *dense for the first half of
training*, so its peak memory equals dense.

**SNIP** and **GraSP** prune once at initialisation and never change the topology. They are the cheapest
methods by peak memory after the single-batch saliency pass, and the weakest at high sparsity, which is
exactly the trade-off the hybrid methods exist to resolve.


In [ ]:
class Method:
    """Base class. All hooks are optional; the shared loop calls whichever exist."""
    key: str = "base"
    label: str = "Base"
    colour: str = "#666"
    is_sparse: bool = True

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.events: List[dict] = []          # phase-transition log

    # -- lifecycle hooks ---------------------------------------------------
    def setup(self, sm, cfg, loader, opt): pass
    def on_epoch_start(self, sm, cfg, epoch, opt, ctx): pass
    def on_after_backward(self, sm, cfg, step, opt, ctx): pass   # grads still dense here
    def on_step_end(self, sm, cfg, step, opt, ctx): pass
    def extra_loss(self, sm, cfg, ctx): return None
    def finalize(self, sm, cfg, opt): pass

    # -- helper ------------------------------------------------------------
    def log_event(self, epoch, block, kind):
        self.events.append({"epoch": epoch, "block": block, "kind": kind})


class DenseMethod(Method):
    key, label, colour, is_sparse = "dense", "Dense (unpruned)", PALETTE["dense"], False

    def setup(self, sm, cfg, loader, opt):
        sm.set_all_dense()


class GMPMethod(Method):
    key, label, colour = "gmp", "GMP (gradual magnitude)", PALETTE["gmp"]

    def setup(self, sm, cfg, loader, opt):
        sm.set_all_dense()
        self.t0 = int(0.15 * cfg.epochs)
        self.t1 = int(0.75 * cfg.epochs)

    def on_epoch_start(self, sm, cfg, epoch, opt, ctx):
        if epoch < self.t0:
            return
        if epoch >= self.t1:
            s = cfg.target_sparsity
        else:
            frac = (epoch - self.t0) / max(self.t1 - self.t0, 1)
            s = cfg.target_sparsity * (1 - (1 - frac) ** 3)      # cubic ramp
        masks = topk_mask_from_scores(score_magnitude(sm), sm.handles, 1 - s,
                                      cfg.sparsity_scope, cfg.min_layer_density)
        apply_masks(sm, masks)
        mask_momentum(opt, sm, sm.handles)


class SingleShotMethod(Method):
    """SNIP / GraSP: score once at initialisation, then never change the topology."""
    criterion = "snip"

    def setup(self, sm, cfg, loader, opt):
        scores = SALIENCY[self.criterion](sm, loader=loader, device=DEVICE, n_batches=10)
        masks = topk_mask_from_scores(scores, sm.handles, 1 - cfg.target_sparsity,
                                      cfg.sparsity_scope, cfg.min_layer_density)
        apply_masks(sm, masks)
        sm.model.zero_grad(set_to_none=True)


class SNIPMethod(SingleShotMethod):
    key, label, colour, criterion = "snip", "SNIP (Lee et al. 2019)", PALETTE["snip"], "snip"


class GraSPMethod(SingleShotMethod):
    key, label, colour, criterion = "grasp", "GraSP (Wang et al. 2020)", PALETTE["grasp"], "grasp"

### 9.3 Dynamic sparse training: SET and RigL

Both maintain a fixed parameter budget and periodically rewire. They differ only in the **regrowth
criterion**, which is the point Evci et al. make: SET revives random dormant connections, RigL revives the
dormant connections with the largest dense gradient magnitude.

The drop fraction is cosine-annealed, $\zeta(t)=\frac{\alpha}{2}\left(1+\cos\frac{t\pi}{T}\right)$, exactly
as in the proposal's §3.4.2, and topology updates stop at 75% of training so the final mask has time to
converge.

RigL's regrowth needs $\nabla_{\mathbf{w}}L$ for weights that are currently *masked out*. The
straight-through estimator of §5.2 provides this — but only between `backward()` and the gradient-masking
step, after which dormant entries are zero. RigL therefore snapshots the dense gradient in the
`on_after_backward` hook, *before* masking. Newly grown weights are initialised to zero, as in the paper.

**Memory note.** RigL's dense-gradient requirement is a real cost that the ledger charges honestly: at a
regrowth step the method momentarily needs a dense gradient tensor for the layer being rewired. Because RigL
rewires *all* layers at the same step, this is itself a (smaller, transient) synchronisation bottleneck —
a point worth noting, since it means the phenomenon this proposal targets is not unique to AC/DC.


In [ ]:
class DynamicSparseMethod(Method):
    """Shared machinery for SET and RigL: ERK init + periodic drop/grow."""
    grow_criterion = "random"

    def setup(self, sm, cfg, loader, opt):
        dens = erk_densities(sm.handles, 1 - cfg.target_sparsity)
        g = torch.Generator(device="cpu").manual_seed(cfg.seed)
        masks = {}
        for h in sm.handles:
            k = max(1, int(round(dens[h.name] * h.numel)))
            perm = torch.randperm(h.numel, generator=g)[:k].to(h.weight.device)
            m = torch.zeros(h.numel, device=h.weight.device); m[perm] = 1.0
            masks[h.name] = m.view_as(h.weight)
        apply_masks(sm, masks)
        self.densities = dens

    def _drop_fraction(self, step, total):
        a = self.cfg.dst_drop_fraction
        return 0.5 * a * (1 + math.cos(math.pi * min(step / max(total, 1), 1.0)))

    @torch.no_grad()
    def _rewire(self, sm, cfg, step, opt, ctx, grads):
        zeta = self._drop_fraction(step, ctx["total_steps"])
        if zeta <= 0:
            return
        for h in sm.handles:
            m = h.mask_fwd
            n_active = int(m.sum().item())
            n_drop = int(zeta * n_active)
            if n_drop < 1:
                continue

            # ---- drop: smallest-magnitude active weights ------------------
            w_abs = h.weight.detach().abs().flatten()
            active_idx = m.flatten().nonzero(as_tuple=True)[0]
            drop_idx = active_idx[torch.topk(w_abs[active_idx], n_drop,
                                             largest=False, sorted=False).indices]
            new_m = m.flatten().clone(); new_m[drop_idx] = 0.0

            # ---- grow: random (SET) or largest dense gradient (RigL) ------
            dormant = (new_m == 0).nonzero(as_tuple=True)[0]
            if dormant.numel() == 0:
                continue
            if self.grow_criterion == "gradient" and grads.get(h.name) is not None:
                score = grads[h.name].abs().flatten()[dormant]
                grow_idx = dormant[torch.topk(score, min(n_drop, dormant.numel()),
                                              sorted=False).indices]
            else:
                perm = torch.randperm(dormant.numel(), device=dormant.device)
                grow_idx = dormant[perm[:min(n_drop, dormant.numel())]]

            new_m[grow_idx] = 1.0
            h.set_masks(new_m.view_as(h.weight))
            h.weight.view(-1)[grow_idx] = 0.0         # revived weights start at zero
        sm.zero_pruned()
        mask_momentum(opt, sm, sm.handles)


class SETMethod(DynamicSparseMethod):
    key, label, colour = "set", "SET (Mocanu et al. 2018)", PALETTE["set"]
    grow_criterion = "random"

    def on_epoch_start(self, sm, cfg, epoch, opt, ctx):
        # SET rewires once per epoch, per the original formulation.
        if 0 < epoch < int(cfg.dst_stop_frac * cfg.epochs):
            self._rewire(sm, cfg, ctx["step"], opt, ctx, grads={})


class RigLMethod(DynamicSparseMethod):
    key, label, colour = "rigl", "RigL (Evci et al. 2020)", PALETTE["rigl"]
    grow_criterion = "gradient"
    _grads = None

    def _is_update_step(self, cfg, step, ctx) -> bool:
        return (step > 0 and step % cfg.dst_update_freq == 0
                and step <= cfg.dst_stop_frac * ctx["total_steps"])

    def on_after_backward(self, sm, cfg, step, opt, ctx):
        # Called by the loop BEFORE sm.mask_gradients() (S5.3, step 2). The STE makes
        # w.grad the DENSE gradient, but only until it is masked: reading it later, in
        # on_step_end, sees zeros at every dormant weight, and the regrowth then simply
        # re-selects the weights it has just dropped.
        self._grads = sm.dense_grads() if self._is_update_step(cfg, step, ctx) else None

    def on_step_end(self, sm, cfg, step, opt, ctx):
        if self._grads is None:
            return
        grads, self._grads = self._grads, None
        self._rewire(sm, cfg, step, opt, ctx, grads)

### 9.4 Top-KAST and N:M structured sparsity

**Top-KAST** (Jayakumar et al., 2020) is the strongest "never dense" competitor and the most direct rival to
the proposal's memory claim, because it *guarantees* a sparse backward pass. Forward uses the top-$K$ weights
($A$); backward updates the top-$(K{+}M)$ weights ($B \supset A$); the extra set $B\setminus A$ is what lets
dormant weights recover. An auxiliary $L_2$ penalty on $B$, scaled inversely with density, discourages the
exploration set from growing without bound. Our ledger charges Top-KAST for $|B|$ non-zeros, not $|A|$ —
which is why it sits above the pure-sparse floor in the memory plots.

**N:M** (Zhou et al., 2021) keeps exactly $N$ of every $M$ consecutive weights, recomputed each step with a
straight-through estimator. We use $N{:}M = 1{:}10$ so the sparsity is exactly 90% and comparable to the
others; note that only 2:4 is actually accelerated by current sparse tensor cores, so the *hardware* benefit
of the 1:10 configuration is hypothetical. It is included because the proposal's literature review covers it
and because it bounds how much accuracy a rigid structural constraint costs.


In [ ]:
class TopKASTMethod(Method):
    key, label, colour = "topkast", "Top-KAST (Jayakumar et al. 2020)", PALETTE["topkast"]

    def setup(self, sm, cfg, loader, opt):
        self.d_fwd = 1 - cfg.target_sparsity
        self.d_bwd = min(1.0, self.d_fwd * cfg.topkast_backward_mult)
        self._select(sm, cfg)

    @torch.no_grad()
    def _select(self, sm, cfg):
        scores = score_magnitude(sm)
        m_f = topk_mask_from_scores(scores, sm.handles, self.d_fwd,
                                    cfg.sparsity_scope, cfg.min_layer_density)
        m_b = topk_mask_from_scores(scores, sm.handles, self.d_bwd,
                                    cfg.sparsity_scope, cfg.min_layer_density)
        for h in sm.handles:
            h.set_masks(m_f[h.name], torch.maximum(m_f[h.name], m_b[h.name]))
        sm.zero_pruned()

    def on_step_end(self, sm, cfg, step, opt, ctx):
        if step % cfg.dst_update_freq == 0:
            self._select(sm, cfg)

    def extra_loss(self, sm, cfg, ctx):
        # L2 on the exploration set, scaled by 1/density (Jayakumar et al., eq. 4).
        reg = 0.0
        for h in sm.handles:
            reg = reg + (h.weight * h.mask_bwd).pow(2).sum() / max(self.d_bwd, 1e-6)
        return cfg.topkast_aux_decay * reg


class NMMethod(Method):
    key, label, colour = "nm", "N:M structured (Zhou et al. 2021)", PALETTE["nm"]

    def setup(self, sm, cfg, loader, opt):
        self._apply(sm, cfg)

    @torch.no_grad()
    def _apply(self, sm, cfg):
        for h in sm.handles:
            h.mask_fwd.copy_(nm_structured_mask(h.weight, cfg.nm_n, cfg.nm_m))
            h.mask_bwd.fill_(1.0)     # SR-STE keeps every weight trainable
            h.invalidate()

    def on_step_end(self, sm, cfg, step, opt, ctx):
        # SR-STE: the mask is recomputed every step and the STE lets pruned weights keep
        # receiving gradient, so the structure can reorganise. The consequence -- which
        # the ledger charges honestly -- is that N:M training holds a DENSE weight,
        # gradient and momentum tensor throughout. It is an inference-acceleration
        # technique, not a training-memory technique.
        self._apply(sm, cfg)

### 9.5 STR — soft threshold reparameterisation

STR (Kusupati et al., 2020) is the only method here that does not use a mask at all. The effective weight is

$$\mathbf{W} = \mathcal{S}(\mathbf{V}, s) = \operatorname{sign}(\mathbf{V})\,\max(|\mathbf{V}| - s,\, 0),
\qquad s = \sigma(s_{\text{raw}}),$$

with one learnable $s_{\text{raw}}$ per layer trained by ordinary backpropagation. Weight decay on
$s_{\text{raw}}$ pulls it toward zero, which *raises* the threshold and therefore increases sparsity over
training. Because the resulting sparsity is an emergent property rather than a constraint, a faithful
implementation lands wherever it lands — which makes an iso-sparsity comparison impossible.

We therefore add a **proportional controller** on the threshold weight decay,

$$\lambda_s \leftarrow \lambda_s \cdot \exp\!\big(\kappa\,(s^{\text{target}} - s^{\text{current}})\big),$$

which steers the run to the target sparsity without altering STR's mechanism. This is an adaptation, it is
disclosed here, and §11 reports STR's *achieved* sparsity alongside its accuracy so the reader can judge the
comparison. If the controller ends more than 2 percentage points off target, a final magnitude projection is
applied and flagged in the results table.

**STR's training memory is dense, and the ledger says so.** $\mathbf{V}$ is a full dense tensor for the whole
run; only the *effective* weight is sparse. STR therefore appears in the memory plots at the dense peak, not
near the sparse floor. This is not a quirk of our implementation — it is the same limitation the proposal
identifies for Movement Pruning (§3.6): a method that produces a sparse deployable model is not thereby a
method that trains within a sparse memory budget. Making that distinction visible is one of the reasons STR
is included.


In [ ]:
class STRMethod(Method):
    key, label, colour = "str", "STR (Kusupati et al. 2020)", PALETTE["str"]

    def setup(self, sm, cfg, loader, opt):
        self.s_params = []
        for h in sm.handles:
            s = nn.Parameter(torch.tensor(-9.0, device=h.weight.device))
            h.module.str_s = s
            self.s_params.append(s)
        # Thresholds live in their own group: no momentum coupling with the weights.
        opt.add_param_group({"params": self.s_params, "block": -1, "lr_mult": 1.0,
                             "weight_decay": 0.0, "name": "str_thresholds"})
        self.lambda_s = 1e-4
        self.kappa = 4.0
        sm.set_all_dense()

    @torch.no_grad()
    def on_step_end(self, sm, cfg, step, opt, ctx):
        # STR's forward path uses the soft-threshold directly, so the masks exist purely
        # for the ledger and the reported sparsity -- refreshing every 25 steps is ample.
        # Only mask_fwd is written: mask_bwd stays all-ones because STR trains a fully
        # DENSE V throughout -- sparsity exists only in the effective weight. The ledger
        # therefore charges STR the dense parameter state, which is the honest figure and
        # is exactly the limitation the proposal raises about Movement Pruning (S3.6).
        if step % 25 == 0:
            for h in sm.handles:
                s = torch.sigmoid(h.module.str_s)
                h.mask_fwd.copy_((h.weight.detach().abs() > s).float())
                h.invalidate()
        if step % 50 == 0:
            cur = sm.global_sparsity()
            err = cfg.target_sparsity - cur
            self.lambda_s = float(np.clip(self.lambda_s * math.exp(self.kappa * err),
                                          1e-8, 1e-1))
        for s in self.s_params:                       # explicit decay toward zero
            s.mul_(1.0 - self.lambda_s)

    @torch.no_grad()
    def _refresh_masks(self, sm):
        for h in sm.handles:
            if h.module.str_s is None:
                continue
            s = torch.sigmoid(h.module.str_s)
            h.mask_fwd.copy_((h.weight.detach().abs() > s).float())
            h.invalidate()

    def finalize(self, sm, cfg, opt):
        self._refresh_masks(sm)                # the cached mask may be up to 25 steps old
        achieved = sm.global_sparsity()
        self.projected = abs(achieved - cfg.target_sparsity) > 0.02
        if self.projected:
            masks = topk_mask_from_scores(score_magnitude(sm), sm.handles,
                                          1 - cfg.target_sparsity, cfg.sparsity_scope,
                                          cfg.min_layer_density)
            apply_masks(sm, masks)
            for h in sm.handles:                      # disable soft-threshold path
                h.module.str_s = None

### 9.6 AC/DC — the synchronised hybrid baseline

The reference implementation of Peste et al. (2021), driven by the schedule matrix of §8.1:

* **Decompression.** Masks go to all-ones for *every* layer at the same instant; momentum is reset globally.
  Previously-pruned weights resume from zero and are free to regrow under unconstrained gradients.
* **Compression.** A global magnitude threshold over the whole prunable scope produces the mask; pruned
  weights and their momentum are zeroed.
* **Termination.** The run ends in a long sparse fine-tuning window so the deployed model is compressed.

The global momentum reset at every decompression is essential to AC/DC's stability and is the exact
mechanism the proposal adapts (§4.2.3).


In [ ]:
class ACDCMethod(Method):
    """Alternating Compressed/DeCompressed with fully synchronised phases."""
    key, label, colour = "acdc", "AC/DC (Peste et al. 2021)", PALETTE["acdc"]

    def build_plan(self, cfg) -> PhasePlan:
        return plan_sync_acdc(cfg)

    def setup(self, sm, cfg, loader, opt):
        self.order = list(range(cfg.n_blocks))
        self.plan = self.build_plan(cfg)
        dense0  = [b for b in range(cfg.n_blocks) if self.plan.dense[0, b]]
        sparse0 = [b for b in range(cfg.n_blocks) if not self.plan.dense[0, b]]
        for b in dense0:
            sm.set_all_dense(sm.blocks()[b])
        if sparse0:
            self._compress(sm, cfg, sparse0, opt, epoch=0, log=False)

    # ---------------------------------------------------------------- phases
    @torch.no_grad()
    def _decompress(self, sm, cfg, blocks, opt, epoch):
        layers = [h for b in blocks for h in sm.blocks()[b]]
        sm.set_all_dense(layers)
        if cfg.momentum_reset == "global":
            reset_momentum(opt, [p for p in sm.model.parameters()])
        elif cfg.momentum_reset == "local":
            reset_momentum(opt, [h.weight for h in layers])
        for b in blocks:
            self.log_event(epoch, b, "decompress")

    @torch.no_grad()
    def _compress(self, sm, cfg, blocks, opt, epoch, density=None, log=True):
        density = density if density is not None else 1 - cfg.target_sparsity
        scope_layers = [h for b in blocks for h in sm.blocks()[b]]
        scores = {h.name: h.weight.detach().abs() for h in scope_layers}
        masks = topk_mask_from_scores(scores, scope_layers, density,
                                      cfg.sparsity_scope, cfg.min_layer_density)
        apply_masks(sm, masks, scope_layers)
        mask_momentum(opt, sm, scope_layers)
        if log:
            for b in blocks:
                self.log_event(epoch, b, "compress")

    # ---------------------------------------------------------------- hooks
    def on_epoch_start(self, sm, cfg, epoch, opt, ctx):
        dec = [b for b, k in self.plan.transitions(epoch) if k == "decompress"]
        com = [b for b, k in self.plan.transitions(epoch) if k == "compress"]
        if com:
            self._compress(sm, cfg, com, opt, epoch)
        if dec:
            self._decompress(sm, cfg, dec, opt, epoch)

    def finalize(self, sm, cfg, opt):
        left = [b for b, k in self.plan.final_transitions()]
        if left:
            self._compress(sm, cfg, left, opt, epoch=cfg.epochs)

### 9.7 Async-AC/DC — the proposed method

The proposed protocol is *structurally* the same algorithm with a different schedule matrix, which is the
cleanest possible demonstration that the contribution is the scheduling, not a bundle of unrelated tricks.
Three things change:

1. **The plan** comes from §8.2, so at most one block is dense at a time.
2. **Compression scope is block-local.** The magnitude threshold is computed over the block being compressed,
   never globally — a global threshold would require every layer's statistics simultaneously and would
   reintroduce a synchronisation point. §14.4 measures what this restriction costs.
3. **Momentum reset is block-local.** Only the decompressing block's velocity is discarded; blocks that are
   mid-sparse-phase keep theirs.

### 9.8 Async-AC/DC++ — the stabilised variant

The proposal identifies gradient staleness as the principal risk (§4.6) and proposes localised momentum
resets, with LR warm-up as a fallback. `Async-AC/DC++` enables both fallbacks plus one addition:

* **Block-local LR warm-up** ($\tau_w$ steps). After decompression the block's LR multiplier ramps
  $0.1 \to 1$, because the freshly-revived weights are at zero with no gradient history and a full-size first
  update is pure noise injection.
* **Compression ramp** ($\tau_c$ steps). Instead of snapping to the target density in one step, the block's
  density is annealed $1 \to d^\ast$ cubically over the final $\tau_c$ steps of its dense window. This
  converts a single large discontinuity in the loss into a sequence of small ones. §13 measures the
  resulting reduction in transition shock directly.

The ramp deliberately runs *inside* the closing block's own window rather than after it. Ramping afterwards
would leave that block partially dense while the next block has already decompressed — two non-sparse blocks
at once, which would silently break the peak-memory guarantee of §8.2. Placed inside the window, neither
stabiliser changes the ledger at all: the block is dense during the ramp either way, so the peak is
unaffected and the memory claim is preserved exactly.


In [ ]:
class AsyncACDCMethod(ACDCMethod):
    """Proposed: staggered decompression, block-local thresholds and momentum resets."""
    key, label, colour = "async_acdc", "Async-AC/DC (proposed)", PALETTE["async_acdc"]

    def build_plan(self, cfg) -> PhasePlan:
        return plan_async_acdc(cfg, self.order)

    def setup(self, sm, cfg, loader, opt):
        self.order = self._block_order(sm, cfg, loader)
        self.plan = self.build_plan(cfg)

        # Every block that is not dense in epoch 0 starts from a sparse initialisation.
        init_sparse = [b for b in range(cfg.n_blocks) if not self.plan.dense[0, b]]
        if init_sparse:
            if cfg.async_warmup == "snip":
                self._snip_init(sm, cfg, loader, init_sparse, opt)
            else:
                self._erk_init(sm, cfg, init_sparse)
        for b in range(cfg.n_blocks):
            if self.plan.dense[0, b]:
                sm.set_all_dense(sm.blocks()[b])

    # ------------------------------------------------------------- ordering
    def _block_order(self, sm, cfg, loader) -> List[int]:
        if cfg.async_order == "reverse":
            return list(range(cfg.n_blocks))[::-1]
        if cfg.async_order == "sensitivity":
            scores = score_snip(sm, loader=loader, device=DEVICE, n_batches=4)
            mean = [float(np.mean([scores[h.name].mean().item() for h in blk]))
                    for blk in sm.blocks()]
            return list(np.argsort(mean)[::-1])       # most sensitive block first
        return list(range(cfg.n_blocks))

    # --------------------------------------------------- sparse initialisers
    @torch.no_grad()
    def _erk_init(self, sm, cfg, blocks):
        layers = [h for b in blocks for h in sm.blocks()[b]]
        dens = erk_densities(layers, 1 - cfg.target_sparsity)
        g = torch.Generator(device="cpu").manual_seed(cfg.seed)
        apply_masks(sm, random_masks(layers, dens, g), layers)

    def _snip_init(self, sm, cfg, loader, blocks, opt):
        """Block-local connection sensitivity -- never makes the whole net dense."""
        layers = [h for b in blocks for h in sm.blocks()[b]]
        scores = score_snip(sm, loader=loader, device=DEVICE, n_batches=4)
        sub = {h.name: scores[h.name] for h in layers}
        apply_masks(sm, topk_mask_from_scores(sub, layers, 1 - cfg.target_sparsity,
                                              cfg.sparsity_scope, cfg.min_layer_density),
                    layers)
        sm.model.zero_grad(set_to_none=True)

    # ---------------------------------------------------------------- phases
    @torch.no_grad()
    def _compress(self, sm, cfg, blocks, opt, epoch, density=None, log=True):
        """Block-LOCAL threshold: compress each block against its own magnitudes.

        A globally-computed threshold would need every layer's magnitude statistics at
        the same instant, which is exactly the synchronisation point this protocol
        exists to remove. Confining the threshold to the block keeps the schedule local.
        """
        density = density if density is not None else 1 - cfg.target_sparsity
        for b in blocks:
            layers = sm.blocks()[b]
            scores = {h.name: h.weight.detach().abs() for h in layers}
            masks = topk_mask_from_scores(scores, layers, density,
                                          cfg.sparsity_scope, cfg.min_layer_density)
            apply_masks(sm, masks, layers)
            mask_momentum(opt, sm, layers)
            if log:
                self.log_event(epoch, b, "compress")


class AsyncACDCPlusMethod(AsyncACDCMethod):
    """Proposed + staleness stabilisers: block-local LR warm-up and a compression ramp."""
    key, label, colour = "async_pp", "Async-AC/DC++ (proposed, stabilised)", PALETTE["async_pp"]

    def setup(self, sm, cfg, loader, opt):
        super().setup(sm, cfg, loader, opt)
        self.warmup_left = {b: 0 for b in range(cfg.n_blocks)}

    def _decompress(self, sm, cfg, blocks, opt, epoch):
        super()._decompress(sm, cfg, blocks, opt, epoch)
        for b in blocks:
            self.warmup_left[b] = cfg.block_lr_warmup_steps
        # Apply the starting multiplier NOW. The loop sets each step's LR before the step,
        # and on_step_end only runs after it, so without this the very first update after
        # decompression -- the one the warm-up exists to protect -- ran at the full LR.
        if cfg.block_lr_warmup_steps > 0:
            for g in opt.param_groups:
                if g.get("block", -1) in blocks:
                    g["lr_mult"] = 0.1

    def _closing_blocks(self, epoch: int, cfg: Config) -> List[int]:
        """Blocks that are dense now and sparse in the next epoch."""
        nxt = epoch + 1
        return [b for b in range(cfg.n_blocks)
                if self.plan.dense[epoch, b]
                and (nxt >= self.plan.epochs or not self.plan.dense[nxt, b])]

    def on_step_end(self, sm, cfg, step, opt, ctx):
        # --- block-local learning-rate warm-up after decompression ---------
        for g in opt.param_groups:
            b = g.get("block", -1)
            if b < 0:
                continue
            left, total = self.warmup_left.get(b, 0), max(cfg.block_lr_warmup_steps, 1)
            if left > 0:
                # multiplier for the NEXT step: step k after decompression uses 0.1 + 0.9 k/total
                g["lr_mult"] = 0.1 + 0.9 * (1 - (left - 1) / total)
                self.warmup_left[b] = left - 1
            else:
                g["lr_mult"] = 1.0

        # --- cubic compression ramp, inside the block's own dense window ----
        # Ramping AFTER the window closes would leave the block partially dense while
        # the next block is already decompressed, so two blocks would be non-sparse at
        # once and the peak-memory guarantee of S8.2 would be violated. Ramping over the
        # final steps of the block's own window keeps at most one block non-sparse.
        if cfg.compression_ramp_steps <= 0:
            return
        spe = ctx["steps_per_epoch"]
        # Clamp to a third of the epoch: on short epochs an unclamped ramp would span
        # the entire dense window, leaving the block no time at full density at all.
        ramp = max(1, min(cfg.compression_ramp_steps, spe // 3))
        remaining = spe - (step - ctx["epoch"] * spe) - 1
        if remaining >= ramp:
            return
        frac = 1.0 - remaining / ramp                                   # 0 -> 1
        d_target = 1 - cfg.target_sparsity
        d_now = 1.0 + (d_target - 1.0) * (frac ** 3)
        for b in self._closing_blocks(ctx["epoch"], cfg):
            self._compress(sm, cfg, [b], opt, ctx["epoch"], density=d_now, log=False)

### 9.9 The shared training loop

One loop serves every method. It implements the update order of §5.3 and instruments four things:

* **Loss trace** at every step, used by §13's transition-shock analysis.
* **Memory trace** sampled every `mem_record_every` steps *and* forced immediately after every phase
  transition, so a peak that lasts one step is never missed.
* **FLOP accumulation** using the density-scaled model of §6.5.
* **Wall-clock**, separating pure training time from evaluation and mask-management overhead, so §11.3 can
  report the true scheduling cost.


In [ ]:
@dataclass
class RunResult:
    cfg: Config
    method_key: str
    method_label: str
    top1: float = 0.0
    top5: float = 0.0
    best_top1: float = 0.0
    final_sparsity: float = 0.0
    peak_param_mib: float = 0.0
    mean_param_mib: float = 0.0
    final_param_mib: float = 0.0
    dense_param_mib: float = 0.0
    train_flops: float = 0.0
    epoch_time_s: float = 0.0
    mask_overhead_s: float = 0.0
    total_time_s: float = 0.0
    acc_curve: List[float] = field(default_factory=list)
    loss_curve: List[float] = field(default_factory=list)
    mem_epochs: List[float] = field(default_factory=list)
    mem_mib: List[float] = field(default_factory=list)
    events: List[dict] = field(default_factory=list)
    notes: str = ""

    def row(self) -> dict:
        return {"method": self.method_label, "key": self.method_key,
                "seed": self.cfg.seed, "top1": self.top1, "top5": self.top5,
                "best_top1": self.best_top1, "sparsity": self.final_sparsity,
                "peak_MiB": self.peak_param_mib, "mean_MiB": self.mean_param_mib,
                "final_MiB": self.final_param_mib,
                "peak_vs_dense_%": 100 * self.peak_param_mib / max(self.dense_param_mib, 1e-9),
                "train_PFLOPs": self.train_flops / 1e15,
                "s_per_epoch": self.epoch_time_s, "notes": self.notes}


@torch.no_grad()
def evaluate(model: nn.Module, loader, device) -> Tuple[float, float]:
    model.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = model(x)
        k = min(5, out.shape[1])
        _, pred = out.topk(k, 1, True, True)
        correct = pred.eq(y.view(-1, 1))
        c1 += correct[:, 0].sum().item()
        c5 += correct.any(dim=1).sum().item()
        n += y.numel()
    model.train()
    return 100.0 * c1 / n, 100.0 * c5 / n


def train_run(cfg: Config, method_cls, verbose: bool = True,
              mem_record_every: int = 25) -> RunResult:
    """Train one (method, seed) condition end to end."""
    set_seed(cfg.seed, deterministic=DETERMINISTIC)
    train_loader, test_loader, n_classes = get_dataloaders(cfg)

    # The loader is memoised across methods and so is its generator, which
    # torch.manual_seed does NOT reset -- a standalone Generator keeps its own state.
    # Without this re-seed each method continues the shuffle sequence where the previous
    # one stopped, so batch order (and therefore the result) depends on where a method
    # sits in the sweep. Re-seeding here is what actually delivers the identical data
    # order that the comparison claims.
    if getattr(train_loader, "generator", None) is not None:
        train_loader.generator.manual_seed(cfg.seed)

    sm = SparseModel(build_model(cfg, n_classes), cfg).to(DEVICE)
    assign_blocks(sm, cfg)
    measure_dense_flops(sm)

    opt = build_optimizer(sm, cfg)
    method: Method = method_cls(cfg)
    tracker = MemoryTracker(sm, cfg)

    t_setup = time.perf_counter()
    method.setup(sm, cfg, train_loader, opt)
    overhead = time.perf_counter() - t_setup

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * cfg.epochs
    ctx = {"total_steps": total_steps, "steps_per_epoch": steps_per_epoch,
           "step": 0, "epoch": 0}

    res = RunResult(cfg=cfg, method_key=method.key, method_label=method.label)
    res.dense_param_mib = dense_reference_memory(sm, cfg).total_mib

    step = 0
    train_seconds = 0.0
    t_wall = time.perf_counter()
    sm.model.train()

    for epoch in range(cfg.epochs):
        ctx["epoch"] = epoch; ctx["step"] = step
        t0 = time.perf_counter()
        method.on_epoch_start(sm, cfg, epoch, opt, ctx)
        overhead += time.perf_counter() - t0
        tracker.record(step, float(epoch))                 # capture post-transition peak

        t_ep = time.perf_counter()
        epoch_losses = []              # kept on-device, drained once at the epoch end
        for i, (x, y) in enumerate(train_loader):
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            set_lr(opt, lr_at(cfg, step, total_steps, steps_per_epoch))

            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(sm.model(x), y)
            aux = method.extra_loss(sm, cfg, ctx)
            if aux is not None:
                loss = loss + aux
            loss.backward()

            t_hook = time.perf_counter()
            method.on_after_backward(sm, cfg, step, opt, ctx)   # step 2: grads still dense
            overhead += time.perf_counter() - t_hook

            if getattr(method, "before_step_grad_mask", lambda: True)():
                sm.mask_gradients()                        # step 3 of the update order
            if cfg.grad_clip:
                torch.nn.utils.clip_grad_norm_(sm.model.parameters(), cfg.grad_clip)
            opt.step()
            sm.project_weights()                           # step 5

            epoch_losses.append(loss.detach())

            t1 = time.perf_counter()
            ctx["step"] = step
            method.on_step_end(sm, cfg, step, opt, ctx)
            overhead += time.perf_counter() - t1

            # Both the ledger and the FLOP model read non-zero counts, which forces a
            # host-device sync. Sampling every `mem_record_every` steps (and always
            # immediately after a phase transition, above) keeps the instrumentation
            # from dominating the latency measurements in S11.3.
            if step % mem_record_every == 0:
                tracker.record(step, epoch + i / steps_per_epoch)
                res.train_flops += step_flops(sm) * cfg.batch_size * mem_record_every
            step += 1

        train_seconds += time.perf_counter() - t_ep

        # One host transfer per epoch instead of two per step. Calling .item() inside the
        # loop forces a device synchronisation on every iteration, which serialises the
        # CPU against the GPU and is pure overhead for a model this small.
        if epoch_losses:
            losses = torch.stack(epoch_losses).float().cpu()
            res.loss_curve.extend(losses.tolist())
            running = float(losses.mean())
        else:
            running = float("nan")
        tracker.record(step, float(epoch + 1))          # end-of-epoch state, for the log

        if (epoch + 1) % cfg.eval_every == 0 or epoch == cfg.epochs - 1:
            top1, top5 = evaluate(sm.model, test_loader, DEVICE)
            res.acc_curve.append(top1)
            res.best_top1 = max(res.best_top1, top1)
            if verbose:
                print(f"  epoch {epoch + 1:3d}/{cfg.epochs}  loss {running:.3f}"
                      f"  top1 {top1:5.2f}%  sparsity {sm.global_sparsity():.3f}"
                      f"  param-mem {tracker.trace[-1].total_mib:6.2f} MiB")

    t0 = time.perf_counter()
    method.finalize(sm, cfg, opt)
    overhead += time.perf_counter() - t0
    tracker.record(step, float(cfg.epochs))

    res.top1, res.top5 = evaluate(sm.model, test_loader, DEVICE)
    res.best_top1 = max(res.best_top1, res.top1)
    res.final_sparsity = sm.global_sparsity()
    res.peak_param_mib = tracker.peak_mib
    res.mean_param_mib = tracker.mean_mib
    res.final_param_mib = tracker.final_mib
    res.epoch_time_s = train_seconds / max(cfg.epochs, 1)
    res.mask_overhead_s = overhead
    res.total_time_s = time.perf_counter() - t_wall
    res.mem_epochs, res.mem_mib = [list(a) for a in tracker.series()]
    res.events = method.events
    if getattr(method, "projected", False):
        res.notes = "final magnitude projection applied to reach target sparsity"

    del sm, opt; gc.collect()
    if ON_CUDA:
        torch.cuda.empty_cache()
    return res


METHODS = {
    "dense":      DenseMethod,
    "gmp":        GMPMethod,
    "snip":       SNIPMethod,
    "grasp":      GraSPMethod,
    "set":        SETMethod,
    "rigl":       RigLMethod,
    "topkast":    TopKASTMethod,
    "str":        STRMethod,
    "nm":         NMMethod,
    "acdc":       ACDCMethod,
    "async_acdc": AsyncACDCMethod,
    "async_pp":   AsyncACDCPlusMethod,
}
print(f"{len(METHODS)} methods registered:")
for k, v in METHODS.items():
    print(f"  {k:<11} {v.label}")

---
# 10. Experiment Runner

Every method is run under the **same** base configuration; only method-specific hyper-parameters that have
no analogue in the other methods are overridden (for instance, `Async-AC/DC++`'s stabiliser lengths, which
are zero for every other method by definition).

Results are cached to `results/` keyed by the full configuration hash, so an interrupted notebook resumes
rather than restarting, and so the analysis sections can be re-executed without retraining.


In [ ]:
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# Method-specific overrides. Anything not listed here inherits BASE exactly, so the
# comparison cannot be skewed by an accidental hyper-parameter difference.
METHOD_OVERRIDES: Dict[str, dict] = {
    "async_pp": dict(block_lr_warmup_steps=100, compression_ramp_steps=150),
    "nm":       dict(nm_n=1, nm_m=10),
}

def cfg_for(method_key: str, seed: int, base: Config = None, **extra) -> Config:
    base = base or BASE
    return replace(base, method=method_key, seed=seed,
                   **{**METHOD_OVERRIDES.get(method_key, {}), **extra})


def _result_path(cfg: Config, suffix: str = "") -> str:
    import hashlib
    h = hashlib.md5(json.dumps(asdict(cfg), sort_keys=True, default=str).encode()).hexdigest()[:10]
    return os.path.join(RESULT_DIR, f"{cfg.method}_{cfg.seed}_{h}{suffix}.json")


def save_result(res: RunResult) -> None:
    payload = {"cfg": asdict(res.cfg),
               **{k: v for k, v in asdict(res).items() if k != "cfg"}}
    with open(_result_path(res.cfg), "w") as f:
        json.dump(payload, f)


def load_result(cfg: Config) -> Optional[RunResult]:
    p = _result_path(cfg)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        payload = json.load(f)
    payload.pop("cfg")
    return RunResult(cfg=cfg, **payload)


def run_method(method_key: str, seed: int, base: Config = None,
               use_cache: bool = True, verbose: bool = True, **extra) -> RunResult:
    cfg = cfg_for(method_key, seed, base, **extra)
    if use_cache:
        cached = load_result(cfg)
        if cached is not None:
            print(f"[cache] {cfg.tag()}  ->  top1 {cached.top1:.2f}%")
            return cached
    print(f"\n=== {METHODS[method_key].label}   (seed {seed}) ===")
    t0 = time.perf_counter()
    res = train_run(cfg, METHODS[method_key], verbose=verbose)
    print(f"    done in {time.perf_counter() - t0:.1f}s  |  top1 {res.top1:.2f}%  "
          f"top5 {res.top5:.2f}%  sparsity {res.final_sparsity:.3f}  "
          f"peak {res.peak_param_mib:.2f} MiB")
    save_result(res)
    return res

In [ ]:
# ------------------------------------------------------------------ MAIN SWEEP
# Order matters only for readability of the log.
RUN_ORDER = ["dense", "gmp", "snip", "grasp", "set", "rigl", "topkast", "str", "nm",
             "acdc", "async_acdc", "async_pp"]

# Trim the method set for the smoke preset so the pipeline check stays quick.
if PRESET == "smoke":
    RUN_ORDER = ["dense", "snip", "rigl", "acdc", "async_acdc", "async_pp"]

RESULTS: Dict[Tuple[str, int], RunResult] = {}

_t_all = time.perf_counter()
for key in RUN_ORDER:
    for seed in SEEDS:
        RESULTS[(key, seed)] = run_method(key, seed, verbose=(PRESET != "full"))
print(f"\nTotal sweep wall-clock: {(time.perf_counter() - _t_all) / 60:.1f} min")

---
# 11. Results

### 11.1 Main comparison table

Columns are grouped into the three metric families the proposal specifies (§4.3): predictive accuracy, peak
memory, and training latency.

* `peak_MiB` is the peak parameter-state footprint from the §6 ledger — the quantity the proposed method
  targets.
* `peak_vs_dense_%` expresses that as a percentage of dense training; **this is the headline number**.
* `train_PFLOPs` is theoretical training compute under the density-scaled model of §6.5.


In [ ]:
def results_frame(results: Dict[Tuple[str, int], RunResult]) -> pd.DataFrame:
    df = pd.DataFrame([r.row() for r in results.values()])
    order = {k: i for i, k in enumerate(RUN_ORDER)}
    df["_o"] = df["key"].map(order)
    return df.sort_values(["_o", "seed"]).drop(columns="_o").reset_index(drop=True)


def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    """Mean +/- std across seeds, keeping the memory columns (which are deterministic)."""
    g = df.groupby(["key", "method"], sort=False)
    agg = g.agg(top1_mean=("top1", "mean"), top1_std=("top1", "std"),
                top5_mean=("top5", "mean"),
                sparsity=("sparsity", "mean"),
                peak_MiB=("peak_MiB", "mean"),
                mean_MiB=("mean_MiB", "mean"),
                final_MiB=("final_MiB", "mean"),
                peak_vs_dense=("peak_vs_dense_%", "mean"),
                PFLOPs=("train_PFLOPs", "mean"),
                s_per_epoch=("s_per_epoch", "mean"),
                n=("top1", "size")).reset_index()
    agg["top1_std"] = agg["top1_std"].fillna(0.0)
    return agg


DF = results_frame(RESULTS)
AGG = aggregate(DF)

_show = AGG.copy()
_show["top-1 (%)"] = _show.apply(
    lambda r: f"{r.top1_mean:.2f}" + (f" +/- {r.top1_std:.2f}" if r.n > 1 else ""), axis=1)
_show["sparsity"] = _show["sparsity"].map(lambda v: f"{v:.1%}")
_show["peak (MiB)"] = _show["peak_MiB"].map(lambda v: f"{v:.2f}")
_show["peak / dense"] = _show["peak_vs_dense"].map(lambda v: f"{v:.1f}%")
_show["PFLOPs"] = _show["PFLOPs"].map(lambda v: f"{v:.3f}")
_show["s/epoch"] = _show["s_per_epoch"].map(lambda v: f"{v:.1f}")

print(_show[["method", "top-1 (%)", "sparsity", "peak (MiB)", "peak / dense",
             "PFLOPs", "s/epoch"]].to_string(index=False))

### 11.2 The accuracy / peak-memory Pareto front

The scatter below is the single most informative figure in this notebook. Down-and-right is better: low peak
memory, high accuracy. Three regions matter.

* **Top-right — dense, GMP, AC/DC.** Accurate, but their peak footprint is the full dense cost. GMP and
  AC/DC are *sparse at deployment* and dense during training, which is exactly the failure mode the proposal
  identifies.
* **Bottom-left — SNIP, GraSP, SET, RigL, Top-KAST.** Genuinely memory-bounded throughout training, but they
  never see an unconstrained gradient and pay for it in accuracy at 90% sparsity.
* **The gap between them** is what Async-AC/DC is designed to occupy: dense-quality exploration at a
  memory-bounded peak.

The dashed line marks the *Pareto frontier* — the set of methods not dominated on both axes simultaneously.


In [ ]:
def pareto_front(points: List[Tuple[float, float]]) -> List[int]:
    """Indices of non-dominated points, minimising x and maximising y."""
    idx = sorted(range(len(points)), key=lambda i: (points[i][0], -points[i][1]))
    front, best_y = [], -np.inf
    for i in idx:
        if points[i][1] > best_y + 1e-12:
            front.append(i); best_y = points[i][1]
    return front


fig, ax = plt.subplots(figsize=(7.2, 5.0))
pts, labels, keys = [], [], []
for _, r in AGG.iterrows():
    pts.append((r.peak_MiB, r.top1_mean)); labels.append(r.method); keys.append(r.key)

front = pareto_front(pts)
fx = [pts[i][0] for i in sorted(front, key=lambda i: pts[i][0])]
fy = [pts[i][1] for i in sorted(front, key=lambda i: pts[i][0])]
ax.plot(fx, fy, ls="--", lw=1.0, color="#999", zorder=1, label="Pareto frontier")

for i, (x, y) in enumerate(pts):
    k = keys[i]
    is_ours = k in ("async_acdc", "async_pp")
    ax.scatter(x, y, s=110 if is_ours else 60, marker=MARKERS.get(k, "o"),
               color=colour_for(k), edgecolor="white", linewidth=1.0,
               zorder=4 if is_ours else 3)
    ax.annotate(labels[i].split(" (")[0], (x, y), textcoords="offset points",
                xytext=(7, 4), fontsize=7.5,
                fontweight="bold" if is_ours else "normal",
                color=colour_for(k))

ax.set_xlabel("peak parameter-state memory (MiB)  --  lower is better")
ax.set_ylabel("top-1 accuracy (%)  --  higher is better")
ax.set_title(f"Accuracy vs. peak training memory  ({BASE.arch}, {BASE.dataset}, "
             f"{BASE.target_sparsity:.0%} sparsity)", loc="left")
ax.legend(loc="lower right")
fig.tight_layout(); save_fig(fig, "03_pareto"); plt.show()

### 11.3 Training dynamics, memory traces and latency

Three diagnostics:

1. **Test accuracy per epoch.** The AC/DC family shows the characteristic saw-tooth — accuracy dips at every
   compression and recovers during the following sparse phase. The synchronised variant's dips are global and
   deep; the staggered variant's are shallower because only one block is disturbed at a time.
2. **Measured memory trace.** Recorded from the actual runs (not simulated), sampled every 25 steps and
   forced after every transition.
3. **Seconds per epoch.** The proposal explicitly asks (§4.3.3) whether mask-management overhead inflates
   runtime. The bar chart separates pure training time from scheduling overhead.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.9))

# --- (a) accuracy curves ------------------------------------------------------
ax = axes[0]
for key in RUN_ORDER:
    r = RESULTS[(key, SEEDS[0])]
    if not r.acc_curve:
        continue
    ep = np.arange(1, len(r.acc_curve) + 1) * BASE.eval_every
    lw = 2.0 if key in ("async_acdc", "async_pp", "acdc") else 1.0
    al = 1.0 if key in ("async_acdc", "async_pp", "acdc", "dense") else 0.45
    ax.plot(ep, r.acc_curve, color=colour_for(key), lw=lw, alpha=al,
            label=r.method_label.split(" (")[0])
ax.set_xlabel("epoch"); ax.set_ylabel("top-1 accuracy (%)")
ax.set_title("(a) Test accuracy", loc="left")
ax.legend(ncol=2, fontsize=6.5, loc="lower right")

# --- (b) memory traces --------------------------------------------------------
ax = axes[1]
for key in ["dense", "rigl", "acdc", "async_acdc", "async_pp"]:
    if (key, SEEDS[0]) not in RESULTS:
        continue
    r = RESULTS[(key, SEEDS[0])]
    ax.plot(r.mem_epochs, r.mem_mib, color=colour_for(key), lw=1.4,
            label=f"{r.method_label.split(' (')[0]} (peak {r.peak_param_mib:.1f})")
ax.set_xlabel("epoch"); ax.set_ylabel("parameter-state memory (MiB)")
ax.set_title("(b) Measured memory trace", loc="left"); ax.legend(fontsize=7)

# --- (c) latency --------------------------------------------------------------
ax = axes[2]
ks = [k for k in RUN_ORDER if (k, SEEDS[0]) in RESULTS]
tr = [np.mean([RESULTS[(k, s)].epoch_time_s for s in SEEDS]) for k in ks]
ov = [np.mean([RESULTS[(k, s)].mask_overhead_s / max(BASE.epochs, 1) for s in SEEDS]) for k in ks]
y = np.arange(len(ks))
ax.barh(y, tr, color=[colour_for(k) for k in ks], height=0.62, label="training")
ax.barh(y, ov, left=tr, color="#D9D9D9", height=0.62, label="mask / schedule overhead")
ax.set_yticks(y); ax.set_yticklabels([METHODS[k].label.split(" (")[0] for k in ks], fontsize=7)
ax.invert_yaxis(); ax.set_xlabel("seconds per epoch")
ax.set_title("(c) Training latency", loc="left"); ax.legend(fontsize=7)

fig.tight_layout(); save_fig(fig, "04_dynamics"); plt.show()

_ov = pd.DataFrame({"method": [METHODS[k].label for k in ks],
                    "train s/epoch": np.round(tr, 2),
                    "overhead s/epoch": np.round(ov, 3),
                    "overhead %": np.round(100 * np.array(ov) / np.maximum(np.array(tr), 1e-9), 2)})
print(_ov.to_string(index=False))

### 11.4 Statistical treatment

The proposal's success criterion (§4.3.2) is that the asynchronous method finishes **within one percentage
point** of the synchronised baseline. That is an equivalence claim, not a difference claim, so a
non-significant $t$-test would be the wrong evidence — failing to reject a null of "no difference" is not
evidence of equivalence.

We therefore report both:

* **Welch's $t$-test** against AC/DC (unequal variances, no assumption that the two schedules produce
  equally variable results), with Cohen's $d$ so the *size* of any difference is visible.
* **A TOST equivalence test** at a margin of $\pm 1.0$ percentage point, which is the actual criterion. TOST
  rejects its null — "the difference is at least as large as the margin" — only when the confidence interval
  for the difference lies entirely inside $[-1, +1]$, which is precisely the claim the proposal makes.

With a single seed neither test is defined; the cell reports that honestly rather than producing a number.
Use `PRESET='full'` (three seeds) for inferential results.


In [ ]:
def welch_and_tost(a: np.ndarray, b: np.ndarray, margin: float = 1.0) -> dict:
    """Welch t-test plus a two-one-sided-tests equivalence check at +/- margin."""
    na, nb = len(a), len(b)
    out = {"n_a": na, "n_b": nb, "mean_a": float(np.mean(a)), "mean_b": float(np.mean(b)),
           "diff": float(np.mean(a) - np.mean(b))}
    if na < 2 or nb < 2:
        out.update(p_welch=float("nan"), cohens_d=float("nan"),
                   tost_p=float("nan"), equivalent=None,
                   note="needs >=2 seeds per condition")
        return out

    t, p = stats.ttest_ind(a, b, equal_var=False)
    sa, sb = np.var(a, ddof=1), np.var(b, ddof=1)
    pooled = math.sqrt(((na - 1) * sa + (nb - 1) * sb) / (na + nb - 2))
    se = math.sqrt(sa / na + sb / nb)
    dfree = (sa / na + sb / nb) ** 2 / ((sa / na) ** 2 / (na - 1) + (sb / nb) ** 2 / (nb - 1))

    d = out["diff"]
    p_lo = stats.t.sf((d + margin) / se, dfree)          # H0: diff <= -margin
    p_hi = stats.t.cdf((d - margin) / se, dfree)         # H0: diff >= +margin
    tost_p = max(p_lo, p_hi)

    out.update(p_welch=float(p),
               cohens_d=float(d / pooled) if pooled > 0 else float("nan"),
               tost_p=float(tost_p), equivalent=bool(tost_p < 0.05),
               note="")
    return out


def acc_vector(key: str) -> np.ndarray:
    return np.array([RESULTS[(key, s)].top1 for s in SEEDS if (key, s) in RESULTS])


if "acdc" in RUN_ORDER:
    base_acc = acc_vector("acdc")
    rows = []
    for key in RUN_ORDER:
        if key == "acdc":
            continue
        st = welch_and_tost(acc_vector(key), base_acc, margin=1.0)
        rows.append({"method": METHODS[key].label.split(" (")[0],
                     "top1": f"{st['mean_a']:.2f}",
                     "vs AC/DC": f"{st['diff']:+.2f}",
                     "p (Welch)": "n/a" if math.isnan(st["p_welch"]) else f"{st['p_welch']:.3f}",
                     "Cohen d": "n/a" if math.isnan(st["cohens_d"]) else f"{st['cohens_d']:+.2f}",
                     "TOST p (+/-1pp)": "n/a" if math.isnan(st["tost_p"]) else f"{st['tost_p']:.3f}",
                     "equivalent?": {True: "yes", False: "no", None: "n/a"}[st["equivalent"]],
                     "within 1pp?": "yes" if st["diff"] > -1.0 else "no"})
    print(f"Reference: AC/DC (sync) top-1 = {base_acc.mean():.2f}%"
          + (f" +/- {base_acc.std(ddof=1):.2f}" if len(base_acc) > 1 else " (single seed)"))
    print()
    print(pd.DataFrame(rows).to_string(index=False))
    if len(SEEDS) < 2:
        print("\nNOTE: inferential columns are undefined with one seed. "
              "The 'within 1pp?' column applies the proposal's criterion to the point estimate.")

---
# 12. The Iso-Memory Comparison — the Headline Result

Sections 11.1-11.2 compare methods with unlimited memory, which flatters AC/DC: it is allowed to spend the
full dense footprint. That is not the situation the proposal is about. The research problem is what happens
when the accelerator **cannot** supply that peak.

### 12.1 Framing

Fix a hardware budget $M_{\text{budget}}$. A method is **feasible** iff its peak parameter-state footprint
fits. Then ask: *among the feasible methods, which is the most accurate?*

This changes the ranking qualitatively rather than quantitatively. Below roughly one third of the dense
parameter state:

* Dense, GMP and **AC/DC become infeasible** — they cannot be run at all, at any accuracy.
* SNIP, GraSP, SET, RigL and Top-KAST remain feasible but have no access to unconstrained gradients.
* **Async-AC/DC remains feasible *and* retains dense exploration**, because its peak is
  $M_{\text{sparse}} + \max_b \Delta_b$ rather than $M_{\text{dense}}$.

There is no way for the synchronised schedule to trade accuracy for peak memory: its dense phase is
all-or-nothing. Shortening $\Delta_d$ reduces the *duration* of the peak but not its *height*. This
all-or-nothing property is the structural reason the staggered schedule is not merely an optimisation of
AC/DC but a strictly larger feasible set.

### 12.2 Sweeping the budget

The cell below sweeps $M_{\text{budget}}$ from the fully-sparse floor up to the dense ceiling and plots, at
each budget, the best accuracy achievable by any feasible method.


In [ ]:
budget_lo = AGG["final_MiB"].min() * 0.98
budget_hi = AGG["peak_MiB"].max() * 1.02
budgets = np.linspace(budget_lo, budget_hi, 400)

best_acc, best_key = [], []
for M in budgets:
    feasible = AGG[AGG["peak_MiB"] <= M]
    if len(feasible) == 0:
        best_acc.append(np.nan); best_key.append(None); continue
    row = feasible.loc[feasible["top1_mean"].idxmax()]
    best_acc.append(row.top1_mean); best_key.append(row.key)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.6, 5.6), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(budgets, best_acc, color="#222", lw=1.8, label="best feasible accuracy")
for _, r in AGG.iterrows():
    ax1.scatter(r.peak_MiB, r.top1_mean, s=90 if r.key in ("async_acdc", "async_pp") else 45,
                marker=MARKERS.get(r.key, "o"), color=colour_for(r.key),
                edgecolor="white", zorder=5)
    ax1.annotate(r.method.split(" (")[0], (r.peak_MiB, r.top1_mean),
                 textcoords="offset points", xytext=(6, -9), fontsize=7,
                 color=colour_for(r.key))
ax1.set_ylabel("top-1 accuracy (%)")
ax1.set_title("Best achievable accuracy as a function of the memory budget", loc="left")
ax1.legend(loc="lower right")

# which method wins at each budget
uniq = [k for k in RUN_ORDER if k in set(best_key)]
ymap = {k: i for i, k in enumerate(uniq)}
ax2.scatter(budgets, [ymap.get(k, np.nan) for k in best_key], s=6,
            c=[colour_for(k) if k else "#fff" for k in best_key])
ax2.set_yticks(range(len(uniq)))
ax2.set_yticklabels([METHODS[k].label.split(" (")[0] for k in uniq], fontsize=7)
ax2.set_xlabel("peak parameter-state memory budget (MiB)")
ax2.set_title("winning method", loc="left", fontsize=9)
fig.tight_layout(); save_fig(fig, "05_iso_memory"); plt.show()

In [ ]:
# ---- Feasibility table at three representative hardware budgets --------------
dense_peak = AGG.loc[AGG["key"] == "dense", "peak_MiB"].iloc[0] if "dense" in set(AGG["key"]) \
    else AGG["peak_MiB"].max()
async_peak = AGG.loc[AGG["key"] == "async_acdc", "peak_MiB"].iloc[0]

scenarios = {
    "generous  (100% of dense)": dense_peak,
    "constrained (50% of dense)": 0.50 * dense_peak,
    "tight (= Async-AC/DC peak)": async_peak,
}

rows = []
for name, M in scenarios.items():
    feas = AGG[AGG["peak_MiB"] <= M + 1e-9]
    if len(feas) == 0:
        rows.append({"budget": name, "MiB": f"{M:.1f}", "feasible methods": 0,
                     "best method": "-- none --", "best top-1": "--"}); continue
    best = feas.loc[feas["top1_mean"].idxmax()]
    rows.append({"budget": name, "MiB": f"{M:.1f}",
                 "feasible methods": len(feas),
                 "best method": best.method.split(" (")[0],
                 "best top-1": f"{best.top1_mean:.2f}%",
                 "AC/DC feasible?": "yes" if ("acdc" in set(feas["key"])) else "NO",
                 "dense feasible?": "yes" if ("dense" in set(feas["key"])) else "NO"})

print(pd.DataFrame(rows).to_string(index=False))
print()

# ---- the direct claim --------------------------------------------------------
tight = AGG[AGG["peak_MiB"] <= async_peak + 1e-9]
rival = tight[~tight["key"].isin(["async_acdc", "async_pp"])]
ours = AGG[AGG["key"] == "async_acdc"].iloc[0]
ours_pp = AGG[AGG["key"] == "async_pp"].iloc[0]
acdc = AGG[AGG["key"] == "acdc"].iloc[0]

print("At a budget equal to the proposed method's own peak:")
if len(rival):
    best_rival = rival.loc[rival["top1_mean"].idxmax()]
    print(f"  best competing feasible method : {best_rival.method.split(' (')[0]}"
          f"  ({best_rival.top1_mean:.2f}%)")
    print(f"  Async-AC/DC                    : {ours.top1_mean:.2f}%"
          f"   ({ours.top1_mean - best_rival.top1_mean:+.2f} pp)")
    print(f"  Async-AC/DC++                  : {ours_pp.top1_mean:.2f}%"
          f"   ({ours_pp.top1_mean - best_rival.top1_mean:+.2f} pp)")
print(f"  AC/DC (sync)                   : INFEASIBLE -- needs "
      f"{acdc.peak_MiB / ours.peak_MiB:.2f}x this budget ({acdc.top1_mean:.2f}% if run)")
print()
print(f"Peak memory reduction vs AC/DC   : {100 * (1 - ours.peak_MiB / acdc.peak_MiB):.1f}%")
print(f"Accuracy difference vs AC/DC     : {ours.top1_mean - acdc.top1_mean:+.2f} pp "
      f"(Async), {ours_pp.top1_mean - acdc.top1_mean:+.2f} pp (Async++)")
print(f"Proposal criterion (within 1 pp) : "
      f"{'MET' if ours_pp.top1_mean - acdc.top1_mean > -1.0 else 'NOT MET'} for Async-AC/DC++")

---
# 13. Transition-Shock Analysis

The proposal's central *risk* (§4.6) is that desynchronising layers destabilises optimisation through
gradient staleness. This section tests that risk directly rather than inferring it from final accuracy.

**Definition.** For a phase transition at step $t^\ast$, the *transition shock* is

$$\text{shock} = \frac{\overline{L}_{[t^\ast,\, t^\ast+w)} - \overline{L}_{[t^\ast-w,\, t^\ast)}}
{\overline{L}_{[t^\ast-w,\, t^\ast)}}$$

— the relative jump in mean training loss across the boundary, with a window $w$ of 50 steps. A compression
event that destroys information produces a large positive shock; a well-conditioned one produces a small one.

**The prediction the mechanism makes.** Synchronised compression removes 90% of the weights of the *entire*
network in a single step, so the whole forward path degrades at once and there is no intact sub-network to
absorb the disruption. Staggered compression removes 90% of *one block*; the remaining blocks are unchanged
and can compensate within a few steps. The staggered schedule should therefore show **smaller shocks, more
often** — the same total disruption, spread out. That is a testable, falsifiable prediction, and it is the
reason to expect staggering to be no worse than synchronising despite the staleness risk.


In [ ]:
SHOCK_COLUMNS = ["epoch", "block", "kind", "before", "after", "shock_%"]

def transition_shocks(res: RunResult, steps_per_epoch: int, window: int = 50) -> pd.DataFrame:
    """Relative jump in mean training loss across each phase transition.

    The window is clamped to half an epoch so that short runs (the `smoke` preset) still
    produce a defined comparison instead of silently dropping every event.
    """
    window = max(5, min(window, steps_per_epoch // 2))
    loss = np.asarray(res.loss_curve)
    rows = []
    for ev in res.events:
        t = int(ev["epoch"]) * steps_per_epoch
        if t - window < 0 or t + window > len(loss):
            continue
        before = loss[t - window: t].mean()
        after = loss[t: t + window].mean()
        rows.append({"epoch": ev["epoch"], "block": ev["block"], "kind": ev["kind"],
                     "before": before, "after": after,
                     "shock_%": 100 * (after - before) / max(before, 1e-9)})
    # An explicit column list keeps the frame usable when no event has a full window.
    return pd.DataFrame(rows, columns=SHOCK_COLUMNS)


_spe = len(get_dataloaders(BASE)[0])
shock_frames = {}
for key in ["acdc", "async_acdc", "async_pp"]:
    if (key, SEEDS[0]) in RESULTS:
        shock_frames[key] = transition_shocks(RESULTS[(key, SEEDS[0])], _spe)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))

ax = axes[0]
for key, fr in shock_frames.items():
    comp = fr[fr["kind"] == "compress"]
    if len(comp):
        ax.scatter(comp["epoch"], comp["shock_%"], s=42, color=colour_for(key),
                   marker=MARKERS.get(key, "o"), edgecolor="white",
                   label=METHODS[key].label.split(" (")[0])
ax.axhline(0, color="#999", lw=0.8)
ax.set_xlabel("epoch"); ax.set_ylabel("training-loss shock (%)")
ax.set_title("(a) Shock at each compression event", loc="left"); ax.legend()

ax = axes[1]
ks = list(shock_frames)
data = [shock_frames[k][shock_frames[k]["kind"] == "compress"]["shock_%"].values for k in ks]
data = [d if len(d) else np.array([0.0]) for d in data]
bp = ax.boxplot(data, patch_artist=True, widths=0.5)
# Tick labels are set separately: boxplot's `labels=` was renamed `tick_labels=` in
# Matplotlib 3.9, and this form works on every version.
ax.set_xticks(range(1, len(ks) + 1))
ax.set_xticklabels([METHODS[k].label.split(" (")[0] for k in ks])
for patch, k in zip(bp["boxes"], ks):
    patch.set_facecolor(colour_for(k)); patch.set_alpha(0.65)
for med in bp["medians"]:
    med.set_color("white")
ax.set_ylabel("training-loss shock (%)")
ax.set_title("(b) Distribution of compression shocks", loc="left")
plt.setp(ax.get_xticklabels(), fontsize=7.5)

fig.tight_layout(); save_fig(fig, "06_transition_shock"); plt.show()

summary = []
for k, fr in shock_frames.items():
    comp = fr[fr["kind"] == "compress"]
    summary.append({"method": METHODS[k].label.split(" (")[0],
                    "n compressions": len(comp),
                    "mean shock %": f"{comp['shock_%'].mean():.2f}" if len(comp) else "--",
                    "max shock %": f"{comp['shock_%'].max():.2f}" if len(comp) else "--",
                    "total disruption %": f"{comp['shock_%'].clip(lower=0).sum():.1f}" if len(comp) else "--"})
print(pd.DataFrame(summary).to_string(index=False))
print("\n'total disruption' sums the positive shocks: staggering trades one large shock")
print("for several small ones, so a lower MAXIMUM with a comparable total is the")
print("signature the mechanism predicts.")

---
# 14. Ablations

Four questions, in decreasing order of how cheaply they can be answered.

### 14.1 Block count $B$ (analytical — always runs)

Peak memory falls as $M_{\text{sparse}} + \Delta_{\text{dense}}/B$, so returns diminish quickly: going from
$B=1$ (i.e. synchronous) to $B=2$ removes half the excess, $B=4$ removes three quarters. Against that,
larger $B$ means shorter, more frequent dense windows per block and a longer cycle $C \ge B\Delta_d$, which
gives each block *less contiguous* dense time. The analytical curve below identifies where the memory
returns flatten; the empirical ablation in §14.2 checks whether accuracy has started to degrade by then.

### 14.2-14.4 Empirical ablations (opt-in)

These require additional training runs and are gated behind `RUN_ABLATIONS`. On the `fast` preset each
ablation adds roughly one extra run per configuration.


In [ ]:
# ---- 14.1 analytical: peak memory vs block count -----------------------------
_sm_ab = SparseModel(build_model(BASE, _nc), BASE)
bs = list(range(1, 13))
peaks, deltas = [], []
for B in bs:
    cfg_b = replace(BASE, n_blocks=B, acdc_cycle=max(BASE.acdc_cycle, B * BASE.acdc_dense_len))
    assign_blocks(_sm_ab, cfg_b)
    plan = plan_async_acdc(cfg_b) if B > 1 else plan_sync_acdc(cfg_b)
    tr = simulate_memory_trace(_sm_ab, cfg_b, plan)
    peaks.append(tr.max())
    deltas.append(tr.max() - _sparse_mib)

fig, ax = plt.subplots(figsize=(6.6, 3.4))
ax.plot(bs, peaks, "o-", color=PALETTE["async_acdc"], label="Async-AC/DC peak")
ax.axhline(_tr_sync.max(), ls="--", color=PALETTE["acdc"], lw=1, label="AC/DC (sync) peak")
ax.axhline(_sparse_mib, ls=":", color="#777", lw=1, label="fully-sparse floor")
ideal = [_sparse_mib + (_tr_sync.max() - _sparse_mib) / B for B in bs]
ax.plot(bs, ideal, ls="-.", lw=1, color="#aaa", label="ideal  floor + delta/B")
ax.set_xlabel("number of blocks B"); ax.set_ylabel("peak parameter-state memory (MiB)")
ax.set_title("Peak memory vs. schedule granularity", loc="left"); ax.legend(fontsize=7.5)
fig.tight_layout(); save_fig(fig, "07_ablation_blocks"); plt.show()

print(pd.DataFrame({"B": bs,
                    "peak MiB": np.round(peaks, 2),
                    "excess over sparse floor": np.round(deltas, 2),
                    "reduction vs sync": [f"{100 * (1 - p / _tr_sync.max()):.1f}%" for p in peaks],
                    "efficiency vs ideal": [f"{i / p:.2f}" for i, p in zip(ideal, peaks)]}
                   ).to_string(index=False))

In [ ]:
# ---- 14.2-14.4 empirical ablations (opt-in) ----------------------------------
RUN_ABLATIONS = False        # set True to run; adds ~6 training runs

ABLATIONS = {
    "B = 2 blocks":            dict(n_blocks=2, acdc_cycle=max(BASE.acdc_cycle, 2 * BASE.acdc_dense_len)),
    "B = 4 blocks":            dict(n_blocks=4, acdc_cycle=max(BASE.acdc_cycle, 4 * BASE.acdc_dense_len)),
    "stage partition":         dict(block_partition="stage"),
    "sensitivity ordering":    dict(async_order="sensitivity"),
    "reverse ordering":        dict(async_order="reverse"),
    "no momentum reset":       dict(momentum_reset="none"),
    "SNIP init (no dense warm-up)": dict(async_warmup="snip"),
    "layerwise thresholds":    dict(sparsity_scope="layerwise"),
}

ABL_RESULTS = {}
if RUN_ABLATIONS:
    ref = RESULTS[("async_acdc", SEEDS[0])]
    ABL_RESULTS["reference"] = ref
    for name, over in ABLATIONS.items():
        print(f"\n--- ablation: {name} ---")
        ABL_RESULTS[name] = run_method("async_acdc", SEEDS[0], **over)

    rows = []
    for name, r in ABL_RESULTS.items():
        rows.append({"variant": name, "top-1": f"{r.top1:.2f}",
                     "delta vs ref": f"{r.top1 - ref.top1:+.2f}",
                     "peak MiB": f"{r.peak_param_mib:.2f}",
                     "sparsity": f"{r.final_sparsity:.1%}"})
    print()
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("RUN_ABLATIONS = False -- empirical ablations skipped.")
    print("Set it to True to measure the effect of block count, partitioning scheme,")
    print("block ordering, momentum-reset policy, warm-up strategy and threshold scope.")

---
# 15. Scaling Projection: ResNet-50 on ImageNet

The proposal targets ImageNet-scale validation (§4.1), which is out of reach for a notebook. But the
*memory* result does not require training: the peak parameter-state footprint is a deterministic function of
the layer sizes, the target sparsity and the schedule. It can therefore be computed **exactly** for
ResNet-50 without a single gradient step, and that projection is the honest way to make an ImageNet-scale
claim here.

Two optimiser regimes are shown, because the choice materially changes the stakes:

* **SGD with momentum** ($S=1$): 3 tensors per parameter.
* **Adam** ($S=2$): 4 tensors per parameter — the regime in which large models are actually trained, and
  where the synchronisation bottleneck is correspondingly worse.

The projection also reports the balanced-partition quality for ResNet-50, whose parameter distribution is
far more skewed than a CIFAR ResNet's (`layer4` alone holds ~60% of the weights), making the dynamic
programme of §7 more valuable rather than less.


In [ ]:
def projection_table(model: nn.Module, cfg: Config, n_blocks_list=(1, 2, 3, 4, 6, 8),
                     sparsities=(0.8, 0.9, 0.95), opt_states=(1, 2)) -> pd.DataFrame:
    sm = SparseModel(model, cfg)
    rows = []
    for S in opt_states:
        for sp in sparsities:
            c = replace(cfg, target_sparsity=sp, optimizer_states=S)
            dense_mib = dense_reference_memory(sm, c).total_mib
            floor_mib = fully_sparse_memory(sm, c, 1 - sp).total_mib
            for B in n_blocks_list:
                cB = replace(c, n_blocks=B,
                             acdc_cycle=max(c.acdc_cycle, B * c.acdc_dense_len))
                assign_blocks(sm, cB)
                plan = plan_sync_acdc(cB) if B == 1 else plan_async_acdc(cB)
                peak = simulate_memory_trace(sm, cB, plan).max()
                rows.append({"optimiser": "SGD-M" if S == 1 else "Adam",
                             "sparsity": f"{sp:.0%}", "B": B,
                             "peak GiB": peak / 1024,
                             "dense GiB": dense_mib / 1024,
                             "sparse floor GiB": floor_mib / 1024,
                             "peak / dense": peak / dense_mib,
                             "saving vs dense": f"{100 * (1 - peak / dense_mib):.1f}%"})
    return pd.DataFrame(rows)


from torchvision.models import resnet50
_r50_cfg = replace(BASE, arch="resnet50", n_blocks=4)
_r50 = resnet50(weights=None)
_r50_sm = SparseModel(copy.deepcopy(_r50), _r50_cfg)
print(f"ResNet-50: {sum(p.numel() for p in _r50.parameters()):,} parameters, "
      f"{_r50_sm.prunable_numel:,} prunable in {len(_r50_sm.handles)} layers")

assign_blocks(_r50_sm, replace(_r50_cfg, block_partition="stage"))
_stage_max = max(sum(h.numel for h in blk) for blk in _r50_sm.blocks())
assign_blocks(_r50_sm, _r50_cfg)
_bal_max = max(sum(h.numel for h in blk) for blk in _r50_sm.blocks())
print(f"largest block, stage partition    : {_stage_max:,} ({_stage_max / _r50_sm.prunable_numel:.1%})")
print(f"largest block, balanced partition : {_bal_max:,} ({_bal_max / _r50_sm.prunable_numel:.1%})")
print(f"-> the DP partitioner cuts the dominant block by "
      f"{100 * (1 - _bal_max / _stage_max):.1f}%\n")

PROJ = projection_table(copy.deepcopy(_r50), _r50_cfg)
print(PROJ[PROJ["optimiser"] == "Adam"].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=False)
for ax, opt_name in zip(axes, ["SGD-M", "Adam"]):
    sub = PROJ[PROJ["optimiser"] == opt_name]
    for sp, grp in sub.groupby("sparsity"):
        grp = grp.sort_values("B")
        ax.plot(grp["B"], grp["peak GiB"], "o-", label=f"sparsity {sp}")
    dense_gib = sub["dense GiB"].iloc[0]
    ax.axhline(dense_gib, ls="--", color=PALETTE["dense"], lw=1,
               label=f"dense / AC/DC-sync ({dense_gib:.2f} GiB)")
    ax.set_xlabel("number of blocks B"); ax.set_ylabel("peak parameter state (GiB)")
    ax.set_title(f"ResNet-50, {opt_name}", loc="left"); ax.legend(fontsize=7)
fig.suptitle("Projected peak training memory for ResNet-50 / ImageNet  (B = 1 is synchronous AC/DC)",
             y=1.03, fontsize=10)
fig.tight_layout(); save_fig(fig, "08_resnet50_projection"); plt.show()

_ad = PROJ[(PROJ["optimiser"] == "Adam") & (PROJ["sparsity"] == "90%")].sort_values("B")
b1 = _ad[_ad["B"] == 1]["peak GiB"].iloc[0]
b4 = _ad[_ad["B"] == 4]["peak GiB"].iloc[0]
print(f"ResNet-50, 90% sparsity, Adam:")
print(f"  synchronous AC/DC (B=1) : {b1:.3f} GiB of parameter state")
print(f"  Async-AC/DC (B=4)       : {b4:.3f} GiB   ({100 * (1 - b4 / b1):.1f}% lower)")
print(f"  absolute saving         : {b1 - b4:.3f} GiB per replica")

In [ ]:
# ---- Validate the analytical ledger against real CUDA allocations ------------
_val_sm = SparseModel(build_model(BASE, _nc), BASE).to(DEVICE)
assign_blocks(_val_sm, BASE)

rows = []
for dense_blocks, name in [([], "all sparse"),
                           ([0], "block 0 dense (async peak)"),
                           (list(range(BASE.n_blocks)), "all dense (sync peak)")]:
    r = cuda_allocation_harness(_val_sm, BASE, dense_blocks)
    rows.append({"configuration": name,
                 "analytic MiB": f"{r['analytic_mib']:.3f}",
                 "measured MiB": "n/a (no CUDA)" if not r["measured"] else f"{r['measured_mib']:.3f}",
                 "rel. error": "n/a" if not r["measured"] else f"{r['rel_error_pct']:+.3f}%"})
print(pd.DataFrame(rows).to_string(index=False))

if ON_CUDA:
    act = measure_activation_memory(build_model(BASE, _nc), BASE.batch_size)
    print(f"\nActivation memory (batch {BASE.batch_size}): {act:.1f} MiB -- identical for every")
    print("method, since architecture and batch size are held fixed. The schedule can only")
    print("move the parameter-state term, which is what S6 measures.")
else:
    print("\nNo CUDA device: the empirical validation column is unavailable, but the")
    print("analytical ledger is exact and identical on any machine.")
del _val_sm; gc.collect()

---
# 16. Discussion

### 16.1 Mapping results back to the proposal's objectives

| Proposal objective (§1.3) | Where it is addressed | Outcome |
|---|---|---|
| Formulate a mathematical scheduling algorithm to stagger dense phases | §7 (optimal balanced contiguous partition, solved by DP) and §8.2 (phase-offset schedule with the non-overlap condition $B\Delta_d \le C$) | Complete and exact |
| Implement the logic inside an established hybrid framework | §9.6-9.8 — Async-AC/DC is the *same* class as the AC/DC baseline with a different schedule matrix, block-local thresholds and block-local momentum resets | Complete |
| Measure and compare peak memory during the backward pass | §6 (byte-exact ledger + CUDA validation), §11.3 (measured traces), §15 (ResNet-50 projection) | Complete |
| Evaluate the effect on convergence and top-1 accuracy | §11.1, §11.4 (Welch + TOST equivalence at $\pm1$pp), §13 (transition shock) | Complete |

### 16.2 What the results actually establish

Read the numbers printed above in this order.

1. **The memory claim is structural, not empirical.** §8.5 and §14.1 derive the peak reduction from the
   schedule alone. It does not depend on the dataset, the seed, or how long we train. The only empirical
   question is whether the analytical ledger matches hardware, and §15's harness answers that to within the
   allocator's rounding granularity.
2. **The accuracy claim is the empirical one**, and it is stated as an equivalence, tested as an equivalence
   (§11.4), against the proposal's own $\pm1$ percentage-point criterion. Note that it is tested under a
   handicap: §8.3 shows the staggered schedule receives materially *less* dense exploration than the
   synchronous baseline, because holding one block dense at a time caps the total. Equivalence achieved under
   that handicap is a stronger result than equivalence at parity would be — but a shortfall would be
   partially explained by it, and should not be read purely as evidence of gradient staleness.
3. **The strongest claim is the iso-memory one (§12).** Under a budget that AC/DC cannot satisfy, the
   proposed method is feasible *and* keeps dense-quality exploration, which no other feasible method has.
   This is a claim about the *feasible set*, and it cannot be answered by tuning the synchronous schedule
   harder, because its dense phase is all-or-nothing.

### 16.3 Honest limitations

* **Scale.** All training here is CIFAR-scale on ResNet-20. The ImageNet claim in §15 is a memory
  projection, not a trained result. This is the limitation the proposal's schedule of work resolves in
  September 2026.
* **Framework overhead is real.** As §5.2 and the proposal's §4.6 both note, PyTorch materialises dense
  gradient tensors regardless of masking. The ledger measures what an efficient implementation *would* use.
  This is the standard convention in the sparse-training literature (RigL and Top-KAST report theoretical
  costs for the same reason), but it means the wall-clock and observed-`nvidia-smi` benefits require a custom
  sparse kernel backend that is outside this notebook's scope.
* **Activation memory is untouched.** At large batch sizes activations dominate, and the proposed method does
  nothing about them. It composes with activation checkpointing rather than competing with it.
* **Unstructured sparsity is not hardware-accelerated.** The N:M baseline exists to make this point: the
  wall-clock benefit of unstructured sparsity on current accelerators is limited regardless of which
  algorithm produces it.
* **Single architecture family.** Transformers, whose parameter distribution is far more uniform across
  depth, would likely partition *better* than a ResNet — but that is a hypothesis, not a result.

### 16.4 What a reviewer should attack first

In the interest of the write-up being defensible, the two weakest points are worth naming explicitly:

1. **The exploration handicap** (§8.3). This is the real one. The non-overlap constraint caps the staggered
   schedule at $E-F$ dense block-epochs while the synchronous schedule gets up to $B$ times that, so the
   proposed method is compared against a baseline that receives strictly *more* unconstrained gradient
   exploration. The staggered schedule is optimal subject to its own constraint — it attains the ceiling —
   but that is a statement about the constraint, not a defence of the comparison. Any accuracy shortfall
   should be attributed here first, before reaching for gradient staleness as the explanation. The two
   defensible responses are to report exploration-per-peak-MiB (§8.5), on which the proposed method wins by
   roughly $B\times$, and to run the iso-memory comparison (§12), where the baseline is not admissible at
   all.
2. **Block-local thresholds** (§9.7). Confining the magnitude threshold to a block forfeits some of the
   cross-layer sparsity reallocation that gives global thresholds their edge. The `layerwise thresholds`
   ablation bounds this cost. If it turns out to be large, a running estimate of the global threshold —
   maintained across cycles from each block's most recent statistics — would recover most of it without
   reintroducing a synchronisation point. That is the most promising extension this notebook suggests and
   is not currently implemented.

### 16.5 Reproducing

```
PRESET = 'smoke'   # pipeline check, runs on CPU
PRESET = 'fast'    # every qualitative claim, ~1.5 h on a T4
PRESET = 'full'    # 3 seeds, 160 epochs, inferential statistics
```

Results cache to `results/` and figures to `figures/`. Deleting `results/` forces a clean re-run.


In [ ]:
# ---- Consolidated export -----------------------------------------------------
summary_path = os.path.join(RESULT_DIR, f"summary_{PRESET}.csv")
AGG.to_csv(summary_path, index=False)
PROJ.to_csv(os.path.join(RESULT_DIR, f"resnet50_projection_{PRESET}.csv"), index=False)

print(f"[export] {summary_path}")
print(f"[export] {os.path.join(RESULT_DIR, f'resnet50_projection_{PRESET}.csv')}")
print(f"[export] figures in ./{FIG_DIR}/\n")

_o  = AGG[AGG["key"] == "async_acdc"].iloc[0]
_op = AGG[AGG["key"] == "async_pp"].iloc[0]
_a  = AGG[AGG["key"] == "acdc"].iloc[0]
_d  = AGG[AGG["key"] == "dense"].iloc[0] if "dense" in set(AGG["key"]) else _a

print("=" * 74)
print(f"HEADLINE  --  {BASE.arch} / {BASE.dataset} / {BASE.target_sparsity:.0%} sparsity / "
      f"preset '{PRESET}' / {len(SEEDS)} seed(s)")
print("=" * 74)
print(f"  peak parameter memory   AC/DC {_a.peak_MiB:7.2f} MiB   ->   "
      f"Async {_o.peak_MiB:7.2f} MiB    ({100 * (1 - _o.peak_MiB / _a.peak_MiB):5.1f}% lower)")
print(f"  as a share of dense     AC/DC {_a.peak_vs_dense:6.1f}%      ->   "
      f"Async {_o.peak_vs_dense:6.1f}%")
print(f"  top-1 accuracy          AC/DC {_a.top1_mean:6.2f}%      ->   "
      f"Async {_o.top1_mean:6.2f}%  /  Async++ {_op.top1_mean:6.2f}%")
print(f"  vs the +/-1 pp criterion            "
      f"{'MET' if (_op.top1_mean - _a.top1_mean) > -1.0 else 'NOT MET'}"
      f"  (gap {_op.top1_mean - _a.top1_mean:+.2f} pp)")
print(f"  dense reference         {_d.top1_mean:6.2f}% at {_d.peak_MiB:.2f} MiB")
print("=" * 74)